<div style="background: linear-gradient(120deg,#14213d,#264653,#2a9d8f); padding: 30px; border-radius: 18px; color: white;">
  <h1 style="margin:0; color:white;">Transformers from Scratch</h1>
  <h3 style="margin:8px 0 0; color:#e9f5f2;">Understanding Encoder–Decoder Architecture Through Mathematics, Code, Training, and Visualization</h3>
  <p style="margin:18px 0 0; font-size: 16px;">A hands-on learning laboratory in TensorFlow: build every core Transformer component, train it on an original synthetic sequence-to-sequence dataset, inspect its attention, and evaluate autoregressive generation.</p>
</div>

**Notebook philosophy:** intuition → mathematics → implementation → verification → training → analysis → inference. The model is implemented in this notebook; no pretrained Transformer, Hugging Face model, or built-in multi-head attention layer is used.

> **Scope note:** the generated task benchmark is intentionally small and rule-based. Its purpose is to expose the mechanics of a Transformer—not to demonstrate general-purpose language understanding or to compete with production language models.

## What you will learn

By the end, you will be able to trace an input token from its integer ID through embeddings, positional information, encoder self-attention, decoder masked self-attention, cross-attention, feed-forward layers, and vocabulary logits. You will train the network with teacher forcing, then switch to autoregressive decoding where future target tokens are unavailable.

- Implement scaled dot-product attention and multi-head attention using tensor operations.
- Understand tensor axes, masks, residual paths, normalization, and the original sinusoidal positional encoding.
- Build custom encoder and decoder stacks with inspectable attention weights.
- Generate a reproducible dataset without downloading files or relying on external services.
- Evaluate loss, token accuracy, exact sequence match, task-specific performance, and latency.
- Save figures, metrics, vocabularies, model weights, and training history.

## Contents

- [Environment and configuration](#environment-and-configuration)
- [Sequence modeling and mathematical foundations](#sequence-modeling-and-mathematical-foundations)
- [Synthetic dataset](#synthetic-dataset)
- [Tokenization and padding](#tokenization-and-padding)
- [Transformer architecture](#transformer-architecture)
- [Scaled dot-product attention](#scaled-dot-product-attention)
- [Multi-head attention](#multi-head-attention)
- [Positional encoding](#positional-encoding)
- [Padding and causal masks](#padding-and-causal-masks)
- [Feed-forward, residual connections, and normalization](#feed-forward-residual-connections-and-normalization)
- [Build the encoder, decoder, and full model](#build-the-encoder-decoder-and-full-model)
- [Teacher forcing and masked loss](#teacher-forcing-and-masked-loss)
- [Training](#training)
- [Autoregressive evaluation](#autoregressive-evaluation)
- [Visual analytics](#visual-analytics)
- [Optional ablation experiments](#optional-ablation-experiments)
- [Tests and revision sheet](#tests-and-revision-sheet)

## Environment and configuration

Use a supported Python environment (Python 3.10–3.12 is a practical starting point for the pinned dependency range in the companion `requirements.txt`). In Google Colab, TensorFlow is usually already installed. In a local environment, create a clean virtual environment and install the requirements before running the notebook.

This notebook intentionally fails with a clear setup message if TensorFlow is unavailable; silently skipping model training would make the results misleading. CPU execution is supported. A GPU is helpful but not required. The default dimensions are kept modest to make experimentation practical on a laptop.

In [1]:
from pathlib import Path
import os
import re
import json
import time
import math
import random
import platform
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

try:
    import tensorflow as tf
except ImportError as exc:
    raise ImportError(
        "TensorFlow is required for the from-scratch model. Install the companion requirements.txt "
        "in Python 3.10–3.12, restart the kernel, and run the notebook again."
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

PROJECT_DIR = Path("transformer_from_scratch_outputs")
OUTPUT_DIRS = {
    "root": PROJECT_DIR,
    "images": PROJECT_DIR / "images",
    "plots_html": PROJECT_DIR / "plots_html",
    "metrics": PROJECT_DIR / "metrics",
    "checkpoints": PROJECT_DIR / "checkpoints",
    "model": PROJECT_DIR / "model",
    "data": PROJECT_DIR / "data",
    "logs": PROJECT_DIR / "logs",
}
for folder in OUTPUT_DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

try:
    import kaleido as _kaleido
    STATIC_PLOT_EXPORT_AVAILABLE = True
except Exception:
    STATIC_PLOT_EXPORT_AVAILABLE = False
STATIC_EXPORT_FAILURE_REPORTED = False

def save_plotly_figure(fig, filename):
    global STATIC_PLOT_EXPORT_AVAILABLE, STATIC_EXPORT_FAILURE_REPORTED
    basename = Path(filename).stem
    html_path = OUTPUT_DIRS["plots_html"] / f"{basename}.html"
    fig.write_html(html_path, include_plotlyjs="cdn")
    if STATIC_PLOT_EXPORT_AVAILABLE:
        try:
            fig.write_image(OUTPUT_DIRS["images"] / f"{basename}.png", scale=2)
        except Exception as exc:
            STATIC_PLOT_EXPORT_AVAILABLE = False
            if not STATIC_EXPORT_FAILURE_REPORTED:
                warnings.warn(f"Static Plotly PNG export unavailable; interactive HTML files are still saved. Reason: {exc}")
                STATIC_EXPORT_FAILURE_REPORTED = True
    return html_path

PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN = "[PAD]", "[UNK]", "[BOS]", "[EOS]"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN]
PAD_ID, UNK_ID, BOS_ID, EOS_ID = range(4)

CONFIG = {
    "d_model": 64,
    "num_heads": 4,
    "num_encoder_layers": 1,
    "num_decoder_layers": 1,
    "d_ff": 128,
    "dropout": 0.10,
    "batch_size": 64,
    "epochs": 6,
    "patience": 2,
    "warmup_steps": 300,
    "gradient_clip_norm": 1.0,
    "examples_per_task": 800,
    "max_sequence_length": 48,
}

print(f"Python: {platform.python_version()}")
print(f"TensorFlow: {tf.__version__}")
print(f"Devices: {[device.name for device in tf.config.list_physical_devices()]}")
print(f"GPU available: {bool(tf.config.list_physical_devices('GPU'))}")
print(f"Output directory: {PROJECT_DIR.as_posix()}")

Python: 3.13.15
TensorFlow: 2.21.0
Devices: ['/physical_device:CPU:0', '/physical_device:GPU:0']
GPU available: True
Output directory: transformer_from_scratch_outputs


### Reproducibility and runtime choices

The random seed controls Python, NumPy, and TensorFlow random generators where supported. Deterministic kernels are requested when the installed TensorFlow build supports them, though exact bitwise reproducibility can still depend on device and library versions.

The default model uses a 64-dimensional representation, four heads, and one encoder/decoder layer. This is smaller than the original paper's widely cited base configuration and is chosen to keep a from-scratch training exercise approachable. Once the entire pipeline works, try `d_model=128`, `d_ff=512`, and two layers per stack on a GPU.

## Sequence modeling and mathematical foundations

A sequence task maps an ordered input to an ordered output. Examples include translation, summarization, structured command execution, and question answering. The source and target can have different lengths, so an encoder–decoder model is useful when the whole input should be represented before generating an output.

- **Encoder-only** models produce contextual representations, often for classification or retrieval.
- **Decoder-only** models predict the next token using only the prefix that has already appeared.
- **Encoder–decoder** models encode a source sequence and let a decoder generate a target sequence while attending to the encoded source.

Recurrent models process tokens step by step and can struggle to preserve long-range information. Self-attention gives each position a content-dependent route to other positions and allows many position computations to be parallelized during training. This is a conceptual distinction—not a guarantee that attention is faster in every wall-clock setting. Real performance depends on sequence length, batch size, hardware, memory bandwidth, kernel implementation, and model dimensions. Standard dense self-attention computes a sequence-length-by-sequence-length score matrix, giving a quadratic sequence-length component.

In [2]:
fig = go.Figure()
steps = ["Token A", "Token B", "Token C", "Token D"]
for i, token in enumerate(steps):
    fig.add_trace(go.Scatter(x=[i], y=[1], mode="markers+text", text=[token], textposition="top center",
                             marker=dict(size=34), name="Recurrent step" if i == 0 else None,
                             showlegend=(i == 0)))
    if i < len(steps) - 1:
        fig.add_annotation(x=i + 0.5, y=1, ax=i, ay=1, xref="x", yref="y", axref="x", ayref="y",
                           text="", showarrow=True, arrowhead=3, arrowsize=1.1, arrowwidth=2)
fig.add_annotation(x=1.5, y=0.48, text="Conceptual recurrent dependency: state is passed step by step", showarrow=False,
                   font=dict(size=12))
fig.add_annotation(x=1.5, y=-0.20, text="Self-attention: every position can score every other position in parallel", showarrow=False,
                   font=dict(size=12))
for i in range(4):
    for j in range(4):
        fig.add_shape(type="line", x0=i, y0=-0.75, x1=j, y1=-1.45, line=dict(width=1, color="rgba(42,157,143,0.45)"))
        fig.add_trace(go.Scatter(x=[i], y=[-0.75], mode="markers+text", text=[steps[i]], textposition="bottom center",
                                 marker=dict(size=18), showlegend=False, hovertemplate=f"{steps[i]} can attend across positions<extra></extra>"))
fig.update_layout(title="Conceptual comparison: sequential recurrence vs pairwise self-attention",
                  xaxis=dict(visible=False, range=[-0.8, 3.8]), yaxis=dict(visible=False, range=[-1.9, 1.5]),
                  height=500, template="plotly_white", margin=dict(l=20, r=20, t=70, b=20))
save_plotly_figure(fig, "sequence_processing_comparison.html")
fig.show()

### Tensor shapes: the dimensions you must keep separate

For the common tensor shape `[batch, sequence, features]`:

| Symbol | Meaning | Example value |
|---|---|---:|
| `B` | number of examples in a mini-batch | 64 |
| `S` | source sequence length | varies by batch padding |
| `T` | target sequence length supplied to the decoder | varies by batch padding |
| `D` or `d_model` | width of each token representation | 64 |
| `H` | number of attention heads | 4 |
| `d_head` | per-head feature width, `D / H` | 16 |
| `d_ff` | hidden width of the position-wise feed-forward layer | 128 |
| `V` | vocabulary size | determined from training data |

A batch axis is not a sequence axis. The head axis is not a vocabulary axis. Many attention bugs are dimension-order bugs, so the notebook prints and asserts shapes rather than relying only on intuition.

### Essential mathematical building blocks

- A **vector** is a one-dimensional collection of values; a **matrix** is a two-dimensional collection; a **tensor** generalizes these dimensions.
- The **dot product** combines aligned vector components. Query–key dot products are used as a similarity score.
- **Matrix multiplication** composes projections and aggregates information across positions. Transposition swaps axes, allowing keys to line up as columns in `QKᵀ`.
- **Softmax** converts a score vector to non-negative values that sum to one: \(\mathrm{softmax}(z_i)=e^{z_i}/\sum_j e^{z_j}\).
- **Embeddings** map discrete token IDs to trainable vectors.
- **Gradient descent** updates parameters to reduce loss. Backpropagation applies the chain rule to compute gradients through the network.
- **Broadcasting** lets compatible dimensions, such as `[B,1,1,S]`, align with `[B,H,T,S]` without explicitly copying data.
- **Masking** excludes forbidden attention links or ignored labels.
- **Residual connections** add a sublayer input back to its output. **Layer normalization** standardizes features within each token representation, then applies learnable scale and offset.
- **Cross-entropy** penalizes low probability assigned to the correct next token. We use raw logits with a numerically stable TensorFlow loss operation.

In [3]:
vector_a = np.array([1.0, 2.0, 3.0])
vector_b = np.array([4.0, 0.5, -1.0])
matrix = np.array([[1.0, 2.0], [3.0, 4.0]])
logits_demo = np.array([1.2, 0.3, -0.4])
softmax_demo = np.exp(logits_demo - logits_demo.max())
softmax_demo = softmax_demo / softmax_demo.sum()

print("Dot product:", float(vector_a @ vector_b))
print("Matrix times its transpose:\n", matrix @ matrix.T)
print("Softmax probabilities:", np.round(softmax_demo, 4), "sum =", round(float(softmax_demo.sum()), 6))
print("Example tensor layout [batch, sequence, features]:", (2, 5, 8))

fig = px.bar(x=["A", "B", "C"], y=softmax_demo, color=softmax_demo,
             labels={"x": "Candidate", "y": "Probability", "color": "Probability"},
             title="A small softmax example: scores become a probability distribution",
             color_continuous_scale="Viridis")
fig.update_layout(template="plotly_white", yaxis_range=[0, 1])
save_plotly_figure(fig, "softmax_numeric_example.html")
fig.show()

Dot product: 2.0
Matrix times its transpose:
 [[ 5. 11.]
 [11. 25.]]
Softmax probabilities: [0.6217 0.2528 0.1255] sum = 1.0
Example tensor layout [batch, sequence, features]: (2, 5, 8)


## Synthetic dataset

The task is **instruction-to-response generation in a controlled mini-language**. Each instruction is generated together with a deterministic answer rule. The data covers arithmetic, list transformations and extraction, comparisons, parity, and conversion of a structured description into a short natural-language response.

This is a deliberately bounded benchmark. It gives the model repeated patterns it can learn and makes each prediction auditable. It is not a substitute for a real translation corpus, nor evidence that a model has learned general arithmetic or broad language understanding.

In [4]:
def format_number_list(values):
    return ", ".join(str(value) for value in values)


def generate_synthetic_dataset(examples_per_task=800, seed=42):
    rng = random.Random(seed)
    rows = []
    seen_sources = set()
    task_names = ["sum", "difference", "reverse", "sort", "maximum", "minimum",
                  "first", "last", "copy", "double", "parity", "compare", "describe"]

    def add_row(task, source, target):
        source = re.sub(r"\s+", " ", source.strip().lower())
        target = re.sub(r"\s+", " ", target.strip().lower())
        if source in seen_sources:
            return False
        seen_sources.add(source)
        rows.append({"example_id": len(rows), "task_type": task, "source_text": source, "target_text": target})
        return True

    for task in task_names:
        accepted = 0
        attempts = 0
        while accepted < examples_per_task and attempts < examples_per_task * 100:
            attempts += 1
            if task == "sum":
                a, b = rng.randint(0, 19), rng.randint(0, 19)
                template = rng.choice(["add {a} and {b}", "what is {a} plus {b}",
                                       "calculate {a} plus {b}", "find the sum of {a} and {b}"])
                source, target = template.format(a=a, b=b), f"sum is {a + b} ."
            elif task == "difference":
                a, b = rng.randint(0, 29), rng.randint(0, 29)
                if a < b:
                    a, b = b, a
                template = rng.choice(["subtract {b} from {a}", "what is {a} minus {b}",
                                       "calculate {a} minus {b}", "find the difference of {a} and {b}"])
                source, target = template.format(a=a, b=b), f"difference is {a - b} ."
            elif task == "reverse":
                values = [rng.randint(0, 9) for _ in range(rng.randint(3, 5))]
                rendered = format_number_list(values)
                template = rng.choice(["reverse the sequence {values}", "write {values} in reverse order",
                                       "reverse these values {values}"])
                source, target = template.format(values=rendered), f"reversed sequence is {format_number_list(values[::-1])} ."
            elif task == "sort":
                values = [rng.randint(0, 9) for _ in range(rng.randint(3, 5))]
                rendered = format_number_list(values)
                template = rng.choice(["sort the sequence {values} ascending", "put {values} in increasing order",
                                       "sort these values {values}"])
                source, target = template.format(values=rendered), f"sorted sequence is {format_number_list(sorted(values))} ."
            elif task in {"maximum", "minimum", "first", "last", "copy"}:
                values = [rng.randint(0, 9) for _ in range(rng.randint(3, 5))]
                rendered = format_number_list(values)
                if task == "maximum":
                    template = rng.choice(["find the largest value in {values}", "what is the maximum of {values}"])
                    answer = f"largest value is {max(values)} ."
                elif task == "minimum":
                    template = rng.choice(["find the smallest value in {values}", "what is the minimum of {values}"])
                    answer = f"smallest value is {min(values)} ."
                elif task == "first":
                    template = rng.choice(["find the first value in {values}", "extract the first item from {values}"])
                    answer = f"first value is {values[0]} ."
                elif task == "last":
                    template = rng.choice(["find the last value in {values}", "extract the last item from {values}"])
                    answer = f"last value is {values[-1]} ."
                else:
                    template = rng.choice(["copy the sequence {values}", "repeat these values {values}"])
                    answer = f"copied sequence is {rendered} ."
                source, target = template.format(values=rendered), answer
            elif task == "double":
                a = rng.randint(0, 99)
                template = rng.choice([
                    "double {a}", "multiply {a} by two", "what is twice {a}",
                    "calculate two times {a}", "find double of {a}", "give me twice {a}",
                    "compute {a} multiplied by two", "return two times number {a}",
                    "scale {a} by two", "what do you get when {a} is doubled"
                ])
                source, target = template.format(a=a), f"double is {2 * a} ."
            elif task == "parity":
                a = rng.randint(0, 99)
                template = rng.choice([
                    "is {a} even or odd", "tell me the parity of {a}", "classify number {a}",
                    "check whether {a} is even", "check whether {a} is odd", "does {a} have even parity",
                    "does {a} have odd parity", "identify the parity for {a}",
                    "say if number {a} is even or odd", "determine whether number {a} is even"
                ])
                source, target = template.format(a=a), f"number is {'even' if a % 2 == 0 else 'odd'} ."
            elif task == "compare":
                a, b = rng.randint(0, 99), rng.randint(0, 99)
                if a == b:
                    continue
                template = rng.choice(["compare {a} and {b}", "which is larger {a} or {b}",
                                       "find the larger number between {a} and {b}"])
                source, target = template.format(a=a, b=b), f"larger value is {max(a, b)} ."
            else:
                color = rng.choice(["red", "blue", "green", "yellow", "black", "white", "orange", "purple"])
                size = rng.choice(["small", "medium", "large"])
                material = rng.choice(["wood", "metal", "plastic", "glass", "stone"])
                shape = rng.choice(["round", "square", "oval", "triangular", "flat", "curved", "cylindrical", "rectangular"])
                template = rng.choice(["describe an item color {color} size {size} material {material} shape {shape}",
                                       "turn this description into a sentence {color} {size} {material} {shape}",
                                       "item details color {color} size {size} material {material} shape {shape}"])
                source = template.format(color=color, size=size, material=material, shape=shape)
                target = f"the item is {color}, {size}, {shape}, and made of {material} ."
            if add_row(task, source, target):
                accepted += 1
        if accepted < examples_per_task:
            raise RuntimeError(f"Could create only {accepted} unique examples for task '{task}'.")

    frame = pd.DataFrame(rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    frame["example_id"] = np.arange(len(frame))
    return frame


dataset = generate_synthetic_dataset(CONFIG["examples_per_task"], SEED)
print(f"Total examples: {len(dataset):,}")
print(f"Unique sources: {dataset['source_text'].nunique():,}")
print("Exact duplicated source rows:", int(dataset["source_text"].duplicated().sum()))
display(dataset.groupby("task_type", sort=False).head(2).sort_values(["task_type", "example_id"]))

Total examples: 10,400
Unique sources: 10,400
Exact duplicated source rows: 0


,example_id,task_type,source_text,target_text
1,1,compare,compare 65 and 2,larger value is 65 .
35,35,compare,find the larger number between 38 and 34,larger value is 38 .
7,7,copy,"repeat these values 3, 9, 8, 7, 6","copied sequence is 3, 9, 8, 7, 6 ."
12,12,copy,"copy the sequence 5, 2, 3, 1","copied sequence is 5, 2, 3, 1 ."
17,17,describe,turn this description into a sentence green sm...,"the item is green, small, cylindrical, and mad..."
21,21,describe,describe an item color orange size medium mate...,"the item is orange, medium, square, and made o..."
25,25,difference,what is 25 minus 1,difference is 24 .
26,26,difference,what is 20 minus 12,difference is 8 .
24,24,double,double 17,double is 34 .
29,29,double,find double of 65,double is 130 .


In [5]:
train_frame, heldout_frame = train_test_split(
    dataset, test_size=0.20, random_state=SEED, stratify=dataset["task_type"]
)
validation_frame, test_frame = train_test_split(
    heldout_frame, test_size=0.50, random_state=SEED, stratify=heldout_frame["task_type"]
)
train_frame = train_frame.reset_index(drop=True)
validation_frame = validation_frame.reset_index(drop=True)
test_frame = test_frame.reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "examples": [len(train_frame), len(validation_frame), len(test_frame)],
})
display(split_summary)
assert set(train_frame.source_text).isdisjoint(set(validation_frame.source_text))
assert set(train_frame.source_text).isdisjoint(set(test_frame.source_text))
assert set(validation_frame.source_text).isdisjoint(set(test_frame.source_text))

fig = px.bar(dataset["task_type"].value_counts().rename_axis("task_type").reset_index(name="examples"),
             x="task_type", y="examples", color="task_type", title="Synthetic dataset: examples per task",
             labels={"task_type": "Task", "examples": "Examples"})
fig.update_layout(template="plotly_white", showlegend=False, xaxis_tickangle=-30)
save_plotly_figure(fig, "dataset_task_distribution.html")
fig.show()

,split,examples
0,train,8320
1,validation,1040
2,test,1040


### Dataset split policy

The data is generated first, then stratified into training, validation, and test sets. Exact source duplicates are removed during generation, so no identical instruction is shared across splits. Vocabulary construction happens only after the split and uses training text only. Since the same controlled templates appear across splits, this tests pattern generalization within the mini-language rather than broad out-of-domain generalization.

A real translation dataset would replace only the source/target table and perhaps the tokenizer. The split discipline, shifting, masking, loss, model, and autoregressive evaluation pipeline would remain largely the same.

## Tokenization and padding

The tokenizer is deliberately lightweight: it separates words, integers, and punctuation. It does not use BPE, WordPiece, SentencePiece, or a pretrained vocabulary. The vocabulary is fitted on **training data only**. Source and target vocabularies are separate, with the same four special tokens reserved at stable IDs.

- `[PAD]` fills unused sequence positions and is excluded from loss and token accuracy.
- `[UNK]` represents a token not seen while fitting the vocabulary.
- `[BOS]` marks the beginning of a decoder target.
- `[EOS]` marks the end of a sequence.

The maximum sequence length is selected from sequence lengths in the generated benchmark and capped by a documented safety limit. It is not used to fit the vocabulary. Truncation is treated as a defensive guard rather than an expected part of these short examples.

In [6]:
TOKEN_PATTERN = re.compile(r"\[PAD\]|\[UNK\]|\[BOS\]|\[EOS\]|-?\d+|[a-z]+|[^\w\s]", flags=re.IGNORECASE)


def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())


class SimpleTokenizer:
    def __init__(self, special_tokens=None):
        self.special_tokens = special_tokens or SPECIAL_TOKENS.copy()
        self.token_to_id = {token: index for index, token in enumerate(self.special_tokens)}
        self.id_to_token = {index: token for token, index in self.token_to_id.items()}
        self.frequencies = Counter()

    def fit(self, texts):
        counts = Counter(token for text in texts for token in tokenize(text))
        for token in self.special_tokens:
            counts.pop(token, None)
        self.frequencies = counts
        for token, _ in sorted(counts.items(), key=lambda item: (-item[1], item[0])):
            if token not in self.token_to_id:
                new_id = len(self.token_to_id)
                self.token_to_id[token] = new_id
                self.id_to_token[new_id] = token
        return self

    def encode(self, text, add_bos=False, add_eos=False):
        ids = [self.token_to_id.get(token, UNK_ID) for token in tokenize(text)]
        if add_bos:
            ids = [BOS_ID] + ids
        if add_eos:
            ids = ids + [EOS_ID]
        return ids

    def decode(self, ids, stop_at_eos=True):
        tokens = []
        for raw_id in ids:
            token_id = int(raw_id)
            if token_id == EOS_ID and stop_at_eos:
                break
            if token_id in (PAD_ID, BOS_ID):
                continue
            tokens.append(self.id_to_token.get(token_id, UNK_TOKEN))
        text = " ".join(tokens)
        text = re.sub(r"\s+([,.!?;:])", r"\1", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

    def to_dict(self):
        return {"token_to_id": self.token_to_id, "frequencies": dict(self.frequencies)}

    @classmethod
    def from_dict(cls, payload):
        instance = cls()
        instance.token_to_id = {str(k): int(v) for k, v in payload["token_to_id"].items()}
        instance.id_to_token = {v: k for k, v in instance.token_to_id.items()}
        instance.frequencies = Counter(payload.get("frequencies", {}))
        return instance


source_tokenizer = SimpleTokenizer().fit(train_frame["source_text"])
target_tokenizer = SimpleTokenizer().fit(train_frame["target_text"])

all_source_lengths = [len(tokenize(text)) + 1 for text in dataset["source_text"]]
all_target_lengths = [len(tokenize(text)) + 2 for text in dataset["target_text"]]
SRC_MAX_LEN = min(max(all_source_lengths), CONFIG["max_sequence_length"])
TGT_MAX_LEN = min(max(all_target_lengths), CONFIG["max_sequence_length"])

print("Source vocabulary size:", len(source_tokenizer.token_to_id))
print("Target vocabulary size:", len(target_tokenizer.token_to_id))
print("Source padded length:", SRC_MAX_LEN)
print("Target padded length:", TGT_MAX_LEN)
print("Special tokens:", source_tokenizer.token_to_id)

Source vocabulary size: 215
Target vocabulary size: 202
Source padded length: 15
Target padded length: 16
Special tokens: {'[PAD]': 0, '[UNK]': 1, '[BOS]': 2, '[EOS]': 3, ',': 4, 'the': 5, '6': 6, '2': 7, '7': 8, '4': 9, 'find': 10, '0': 11, '9': 12, '5': 13, '3': 14, '1': 15, '8': 16, 'in': 17, 'is': 18, 'value': 19, 'what': 20, 'of': 21, 'item': 22, 'and': 23, 'from': 24, 'these': 25, 'values': 26, 'sequence': 27, 'first': 28, 'last': 29, 'reverse': 30, 'extract': 31, 'number': 32, 'order': 33, 'color': 34, 'material': 35, 'shape': 36, 'size': 37, 'larger': 38, 'sort': 39, 'calculate': 40, 'largest': 41, 'even': 42, 'repeat': 43, 'or': 44, 'plus': 45, 'smallest': 46, 'minus': 47, 'two': 48, 'minimum': 49, 'copy': 50, 'maximum': 51, 'parity': 52, 'odd': 53, 'medium': 54, 'increasing': 55, 'put': 56, 'compare': 57, 'details': 58, 'a': 59, 'description': 60, 'into': 61, 'sentence': 62, 'this': 63, 'turn': 64, 'write': 65, 'between': 66, 'large': 67, 'ascending': 68, 'which': 69, 'small'

In [7]:
def encode_and_pad(texts, tokenizer, max_length, add_bos=False, add_eos=False):
    encoded = [tokenizer.encode(text, add_bos=add_bos, add_eos=add_eos) for text in texts]
    clipped = [sequence[:max_length] for sequence in encoded]
    padded = tf.keras.utils.pad_sequences(
        clipped, maxlen=max_length, padding="post", truncating="post", value=PAD_ID, dtype="int32"
    )
    return padded, encoded


train_src, train_src_unpadded = encode_and_pad(train_frame.source_text, source_tokenizer, SRC_MAX_LEN, add_eos=True)
val_src, val_src_unpadded = encode_and_pad(validation_frame.source_text, source_tokenizer, SRC_MAX_LEN, add_eos=True)
test_src, test_src_unpadded = encode_and_pad(test_frame.source_text, source_tokenizer, SRC_MAX_LEN, add_eos=True)

train_tgt_full, train_tgt_unpadded = encode_and_pad(train_frame.target_text, target_tokenizer, TGT_MAX_LEN, add_bos=True, add_eos=True)
val_tgt_full, val_tgt_unpadded = encode_and_pad(validation_frame.target_text, target_tokenizer, TGT_MAX_LEN, add_bos=True, add_eos=True)
test_tgt_full, test_tgt_unpadded = encode_and_pad(test_frame.target_text, target_tokenizer, TGT_MAX_LEN, add_bos=True, add_eos=True)

train_tgt_in, train_labels = train_tgt_full[:, :-1], train_tgt_full[:, 1:]
val_tgt_in, val_labels = val_tgt_full[:, :-1], val_tgt_full[:, 1:]
test_tgt_in, test_labels = test_tgt_full[:, :-1], test_tgt_full[:, 1:]

example_index = 0
example_tokens = tokenize(train_frame.loc[example_index, "source_text"])
example_ids = source_tokenizer.encode(train_frame.loc[example_index, "source_text"], add_eos=True)
example_decoded = source_tokenizer.decode(example_ids)
print("Source sentence:", train_frame.loc[example_index, "source_text"])
print("Tokens:", example_tokens)
print("Token IDs:", example_ids)
print("Reconstructed:", example_decoded)
print("Target sentence:", train_frame.loc[example_index, "target_text"])
assert example_decoded == train_frame.loc[example_index, "source_text"]

Source sentence: find the first value in 5, 7, 8
Tokens: ['find', 'the', 'first', 'value', 'in', '5', ',', '7', ',', '8']
Token IDs: [10, 5, 28, 19, 17, 13, 4, 8, 4, 16, 3]
Reconstructed: find the first value in 5, 7, 8
Target sentence: first value is 5 .


In [8]:
frequency_frame = pd.DataFrame(source_tokenizer.frequencies.most_common(30), columns=["token", "frequency"])
fig = px.bar(frequency_frame.sort_values("frequency"), x="frequency", y="token", orientation="h",
             color="frequency", color_continuous_scale="Tealgrn", title="Most frequent source tokens in training data")
fig.update_layout(template="plotly_white", height=650)
save_plotly_figure(fig, "vocabulary_frequency.html")
fig.show()

length_frame = pd.DataFrame({
    "sequence_length": [len(tokenize(x)) for x in dataset.source_text] + [len(tokenize(x)) for x in dataset.target_text],
    "sequence_type": ["source"] * len(dataset) + ["target"] * len(dataset),
})
fig = px.histogram(length_frame, x="sequence_length", color="sequence_type", barmode="overlay", opacity=0.75,
                   nbins=25, title="Source and target token-length distributions")
fig.update_layout(template="plotly_white", xaxis_title="Number of tokens", yaxis_title="Examples")
save_plotly_figure(fig, "sequence_length_distribution.html")
fig.show()

coverage_rows = []
for split_name, frame in [("validation", validation_frame), ("test", test_frame)]:
    for text_type, texts, tokenizer in [("source", frame.source_text, source_tokenizer), ("target", frame.target_text, target_tokenizer)]:
        total = sum(len(tokenize(text)) for text in texts)
        unknown = sum(token not in tokenizer.token_to_id for text in texts for token in tokenize(text))
        coverage_rows.append({"split": split_name, "text_type": text_type, "tokens": total,
                              "unknown_tokens": unknown, "unknown_rate": unknown / max(total, 1)})
display(pd.DataFrame(coverage_rows))

,split,text_type,tokens,unknown_tokens,unknown_rate
0,validation,source,9202,0,0.0
1,validation,target,7032,0,0.0
2,test,source,9259,0,0.0
3,test,target,7082,0,0.0


### What tokenization changes

Tokenization determines the vocabulary size and the number of positions the Transformer processes. A word-level tokenizer can be compact on a constrained vocabulary but maps unseen words to `[UNK]`; a subword tokenizer usually reduces unknown words but can lengthen sequences. Dense self-attention has a score matrix proportional to the square of sequence length, so a change in tokenization can materially affect compute and memory.

## Transformer architecture

The encoder converts the source sequence into contextual memory. The decoder receives the target prefix shifted right, applies causal self-attention to prevent looking ahead, and uses cross-attention to consult the encoder memory. The final linear layer maps each decoder position to target-vocabulary logits.

```text
SOURCE TOKENS → Embedding + Position → Encoder layer × N → ENCODER MEMORY
                                                              ↓ keys / values
TARGET PREFIX → Embedding + Position → Masked self-attention → Cross-attention
                                                              → Feed-forward
                                                              → Decoder layer × N
                                                              → Linear projection → Vocabulary logits
```

In the implementation, encoder and decoder blocks contain residual connections and layer normalization around the sublayers. The decoder layer order is masked self-attention, cross-attention, then feed-forward network. The source padding mask applies to source keys; the decoder self-attention mask combines target padding and a causal look-ahead mask.

In [9]:
architecture_nodes = {
    "source": (1.8, 9.1, 2.8, 0.65, "Source token IDs", "#dbeafe", "#1d4ed8"),
    "src_embed": (1.8, 8.0, 2.8, 0.65, "Token embedding + position", "#bfdbfe", "#1d4ed8"),
    "encoder": (1.8, 6.5, 2.8, 1.0, "Encoder layer × N\nself-attention → FFN\n(residual + norm)", "#93c5fd", "#1d4ed8"),
    "memory": (1.8, 4.9, 2.8, 0.65, "Encoder memory", "#60a5fa", "#1d4ed8"),
    "target": (8.0, 9.1, 3.0, 0.65, "Shifted target token IDs", "#fce7f3", "#9d174d"),
    "tgt_embed": (8.0, 8.0, 3.0, 0.65, "Token embedding + position", "#fbcfe8", "#9d174d"),
    "decoder": (8.0, 6.0, 3.0, 1.35, "Decoder layer × N\nmasked self-attention\n cross-attention + FFN\n(residual + norm)", "#f9a8d4", "#9d174d"),
    "projection": (8.0, 4.1, 3.0, 0.65, "Linear vocabulary projection", "#f5d0fe", "#86198f"),
    "logits": (8.0, 3.0, 3.0, 0.65, "Vocabulary logits [B, T, V]", "#ddd6fe", "#6d28d9"),
    "output": (8.0, 1.9, 3.0, 0.65, "Next-token prediction", "#ede9fe", "#6d28d9"),
}
fig = go.Figure()
for key, (x, y, w, h, label, fill, line) in architecture_nodes.items():
    fig.add_shape(type="rect", x0=x-w/2, x1=x+w/2, y0=y-h/2, y1=y+h/2,
                  line=dict(color=line, width=2), fillcolor=fill, layer="below")
    fig.add_annotation(x=x, y=y, text=label, showarrow=False, font=dict(size=11, color="#172033"))

def arrow(start, end, color="#475569", width=2, label=None, label_x=None, label_y=None):
    fig.add_annotation(x=end[0], y=end[1], ax=start[0], ay=start[1], xref="x", yref="y", axref="x", ayref="y",
                       text="", showarrow=True, arrowhead=3, arrowsize=1.1, arrowwidth=width, arrowcolor=color)
    if label:
        fig.add_annotation(x=label_x if label_x is not None else (start[0]+end[0])/2,
                           y=label_y if label_y is not None else (start[1]+end[1])/2,
                           text=label, showarrow=False, font=dict(size=10, color=color), bgcolor="white")

arrow((1.8, 8.76), (1.8, 8.34))
arrow((1.8, 7.66), (1.8, 7.02))
arrow((1.8, 6.0), (1.8, 5.24))
arrow((8.0, 8.76), (8.0, 8.34))
arrow((8.0, 7.66), (8.0, 6.69))
arrow((8.0, 5.31), (8.0, 4.44))
arrow((8.0, 3.76), (8.0, 3.34))
arrow((8.0, 2.66), (8.0, 2.24))
arrow((3.25, 4.9), (6.5, 6.22), color="#0f766e", width=3, label="memory as K, V", label_x=4.85, label_y=5.8)
fig.add_annotation(x=1.8, y=3.7, text="ENCODER", showarrow=False, font=dict(size=13, color="#1d4ed8"))
fig.add_annotation(x=8.0, y=0.95, text="DECODER", showarrow=False, font=dict(size=13, color="#9d174d"))
fig.update_layout(title="Encoder–decoder Transformer: information flow", template="plotly_white", width=1000, height=850,
                  xaxis=dict(visible=False, range=[-0.3, 11.7]), yaxis=dict(visible=False, range=[0.4, 9.8]),
                  margin=dict(l=20, r=20, t=70, b=20), showlegend=False)
save_plotly_figure(fig, "architecture_diagram.html")
fig.show()

### Shape and parameter intuition

The encoder memory has shape `[B, S, D]`. The decoder representation has shape `[B, T, D]`. Multi-head attention reshapes the feature axis from `D` to `[H, d_head]`, computes attention for every head, then concatenates the head outputs back to `D`. The output projection has shape `[B, T, V_target]` and contains logits, not probabilities.

The attention matrix for self-attention has shape `[B, H, S, S]` in the encoder and `[B, H, T, T]` in the decoder. Cross-attention has shape `[B, H, T, S]` because every target position can score each source position.

## Scaled dot-product attention

For queries \(Q\), keys \(K\), and values \(V\), the original Transformer uses

\[
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}} + M\right)V,
\]

where \(M\) represents masking (forbidden positions receive a very negative score). Queries ask what a position needs, keys describe what positions offer for matching, and values carry the information to aggregate. The terms are learned projections of token representations; these labels are a useful interpretation, not literal human questions.

The division by \(\sqrt{d_k}\) keeps dot-product magnitude from growing too much as feature dimension increases. Large logits can push softmax toward extremely sharp distributions and reduce useful gradients. The code below returns raw scores, scaled/masked scores, attention probabilities, and weighted values so each stage can be inspected.

In [10]:
def scaled_dot_product_attention(query, key, value, mask=None):
    query = tf.convert_to_tensor(query)
    key = tf.convert_to_tensor(key)
    value = tf.convert_to_tensor(value)
    raw_scores = tf.matmul(query, key, transpose_b=True)
    depth = tf.cast(tf.shape(key)[-1], raw_scores.dtype)
    scaled_scores = raw_scores / tf.math.sqrt(depth)
    if mask is not None:
        mask = tf.cast(mask, tf.bool)
        scaled_scores = tf.where(mask, tf.cast(-1e9, scaled_scores.dtype), scaled_scores)
    attention_weights = tf.nn.softmax(scaled_scores, axis=-1)
    weighted_values = tf.matmul(attention_weights, value)
    return weighted_values, attention_weights, raw_scores, scaled_scores


q_demo = tf.constant([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]])
k_demo = tf.constant([[[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]]])
v_demo = tf.constant([[[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]]])
weighted_demo, weights_demo, raw_demo, scaled_demo = scaled_dot_product_attention(q_demo, k_demo, v_demo)
print("Q shape:", q_demo.shape, "K shape:", k_demo.shape, "V shape:", v_demo.shape)
print("Raw scores:\n", raw_demo.numpy()[0])
print("Scaled scores:\n", scaled_demo.numpy()[0])
print("Attention weights:\n", np.round(weights_demo.numpy()[0], 4))
print("Weighted values:\n", np.round(weighted_demo.numpy()[0], 3))
assert weighted_demo.shape == (1, 3, 2)
assert np.allclose(weights_demo.numpy().sum(axis=-1), 1.0, atol=1e-6)

Q shape: (1, 3, 2) K shape: (1, 3, 2) V shape: (1, 3, 2)
Raw scores:
 [[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 2.]]
Scaled scores:
 [[0.70710677 0.         0.70710677]
 [0.         0.70710677 0.70710677]
 [0.70710677 0.70710677 1.4142135 ]]
Attention weights:
 [[0.4011 0.1978 0.4011]
 [0.1978 0.4011 0.4011]
 [0.2483 0.2483 0.5035]]
Weighted values:
 [[ 2.    20.   ]
 [ 2.203 22.033]
 [ 2.255 22.552]]


In [11]:
def attention_heatmap(matrix, x_labels, y_labels, title, filename, colorscale="Viridis", zmin=None, zmax=None):
    matrix = np.asarray(matrix)
    fig = go.Figure(data=go.Heatmap(z=matrix, x=x_labels, y=y_labels, colorscale=colorscale,
                                    zmin=zmin, zmax=zmax, colorbar=dict(title="Value"),
                                    hovertemplate="row=%{y}<br>column=%{x}<br>value=%{z:.4f}<extra></extra>"))
    fig.update_layout(title=title, template="plotly_white", xaxis_title="Key / source position", yaxis_title="Query position",
                      width=760, height=560, yaxis_autorange="reversed")
    save_plotly_figure(fig, filename)
    fig.show()
    return fig

query_labels = ["query 1", "query 2", "query 3"]
key_labels = ["key 1", "key 2", "key 3"]
attention_heatmap(raw_demo.numpy()[0], key_labels, query_labels, "Raw query–key dot products", "attention_raw_scores.html", "RdBu")
attention_heatmap(scaled_demo.numpy()[0], key_labels, query_labels, "Scaled attention scores", "attention_scaled_scores.html", "RdBu")
attention_heatmap(weights_demo.numpy()[0], key_labels, query_labels, "Normalized attention probabilities", "attention_probabilities.html", "Viridis", 0, 1)

fig = go.Figure()
for i in range(weighted_demo.shape[1]):
    fig.add_trace(go.Bar(name=f"Query {i+1}", x=["Feature 1", "Feature 2"], y=weighted_demo.numpy()[0, i],
                         offsetgroup=str(i)))
fig.update_layout(title="Weighted values after attention aggregation", barmode="group", template="plotly_white",
                  yaxis_title="Aggregated value")
save_plotly_figure(fig, "attention_weighted_values.html")
fig.show()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Without scaling", "With scaling"))
raw_weights = tf.nn.softmax(raw_demo, axis=-1).numpy()[0]
scaled_weights = tf.nn.softmax(scaled_demo, axis=-1).numpy()[0]
fig.add_trace(go.Heatmap(z=raw_weights, x=key_labels, y=query_labels, zmin=0, zmax=1, colorscale="Cividis"), row=1, col=1)
fig.add_trace(go.Heatmap(z=scaled_weights, x=key_labels, y=query_labels, zmin=0, zmax=1, colorscale="Cividis"), row=1, col=2)
fig.update_layout(title="Scaling changes the sharpness of the softmax distribution", template="plotly_white", height=450)
save_plotly_figure(fig, "attention_scaling_comparison.html")
fig.show()

## Multi-head attention

A multi-head layer learns separate query, key, and value projections for each head. Each head performs scaled dot-product attention, and the resulting feature blocks are concatenated and passed through a final projection. With model width `D` and `H` heads, `d_head = D / H`; therefore `D` must be divisible by `H`.

Heads can learn different relationships, but one should not claim that each head necessarily has a clean, human-interpretable linguistic role. That requires empirical inspection and may vary across training runs.

In [12]:
class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, **kwargs):
        super().__init__(**kwargs)
        if d_model % num_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by num_heads ({num_heads}).")
        self.d_model = int(d_model)
        self.num_heads = int(num_heads)
        self.depth = self.d_model // self.num_heads
        self.query_projection = tf.keras.layers.Dense(self.d_model, use_bias=True, name="query_projection")
        self.key_projection = tf.keras.layers.Dense(self.d_model, use_bias=True, name="key_projection")
        self.value_projection = tf.keras.layers.Dense(self.d_model, use_bias=True, name="value_projection")
        self.output_projection = tf.keras.layers.Dense(self.d_model, use_bias=True, name="output_projection")

    def split_heads(self, tensor):
        batch_size = tf.shape(tensor)[0]
        sequence_length = tf.shape(tensor)[1]
        tensor = tf.reshape(tensor, [batch_size, sequence_length, self.num_heads, self.depth])
        return tf.transpose(tensor, [0, 2, 1, 3])

    def call(self, query, key, value, mask=None, training=False):
        q = self.split_heads(self.query_projection(query))
        k = self.split_heads(self.key_projection(key))
        v = self.split_heads(self.value_projection(value))
        head_output, attention_weights, _, _ = scaled_dot_product_attention(q, k, v, mask=mask)
        head_output = tf.transpose(head_output, [0, 2, 1, 3])
        batch_size = tf.shape(head_output)[0]
        query_length = tf.shape(head_output)[1]
        concatenated = tf.reshape(head_output, [batch_size, query_length, self.d_model])
        output = self.output_projection(concatenated)
        return output, attention_weights


mha_demo = MultiHeadAttention(d_model=8, num_heads=2)
random_tensor = tf.random.normal([2, 5, 8], seed=SEED)
mha_output, mha_weights = mha_demo(random_tensor, random_tensor, random_tensor)
print("Input:", random_tensor.shape)
print("Head dimension:", mha_demo.depth)
print("Output:", mha_output.shape)
print("Attention weights:", mha_weights.shape)
assert mha_output.shape == (2, 5, 8)
assert mha_weights.shape == (2, 2, 5, 5)
assert 8 % 2 == 0

Input: (2, 5, 8)
Head dimension: 4
Output: (2, 5, 8)
Attention weights: (2, 2, 5, 5)


The shape path for one self-attention batch is:

`[B, S, D] → linear projection [B, S, D] → split heads [B, H, S, d_head] → scores [B, H, S, S] → context [B, H, S, d_head] → concatenate [B, S, D] → output projection [B, S, D]`.

Cross-attention uses target states for queries and encoder memory for keys and values, so the score tensor becomes `[B, H, T, S]`.

## Positional encoding

Self-attention alone does not force an order on the input: if token representations are permuted, attention can process the permuted set without knowing which token originally came first. The original Transformer therefore adds positional information to token embeddings.

The sinusoidal encoding is

\[
PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/D}}\right), \qquad
PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/D}}\right).
\]

`pos` is the sequence position, `i` indexes dimension pairs, and the wavelengths vary by feature dimension. Token embeddings encode learned token identity; positional encodings supply a deterministic location-dependent pattern. In this notebook, positions are computed from TensorFlow operations and are not trainable parameters.

In [13]:
def sinusoidal_position_encoding(max_length, d_model):
    positions = tf.cast(tf.range(max_length)[:, tf.newaxis], tf.float32)
    dimensions = tf.cast(tf.range(d_model)[tf.newaxis, :], tf.float32)
    paired_dimension = 2.0 * tf.floor(dimensions / 2.0)
    angle_rates = tf.pow(10000.0, -paired_dimension / tf.cast(d_model, tf.float32))
    angles = positions * angle_rates
    even_dimension = tf.equal(tf.math.floormod(tf.range(d_model), 2), 0)[tf.newaxis, :]
    encoding = tf.where(even_dimension, tf.sin(angles), tf.cos(angles))
    return encoding[tf.newaxis, :, :]


class SinusoidalPositionEncoding(tf.keras.layers.Layer):
    def __init__(self, max_length, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_length = int(max_length)
        self.d_model = int(d_model)
        self.encoding = sinusoidal_position_encoding(self.max_length, self.d_model)

    def call(self, inputs):
        sequence_length = tf.shape(inputs)[1]
        position_slice = self.encoding[:, :sequence_length, :]
        return inputs + tf.cast(position_slice, inputs.dtype)


position_matrix = sinusoidal_position_encoding(80, 64).numpy()[0]
fig = px.imshow(position_matrix.T, aspect="auto", origin="lower", color_continuous_scale="RdBu_r",
                labels={"x": "Position", "y": "Embedding dimension", "color": "Encoding"},
                title="Sinusoidal positional encoding across positions and dimensions")
fig.update_layout(template="plotly_white", width=1000, height=600)
save_plotly_figure(fig, "positional_encoding_heatmap.html")
fig.show()

fig = go.Figure()
for dimension in [0, 1, 4, 5, 16, 17]:
    fig.add_trace(go.Scatter(x=np.arange(80), y=position_matrix[:, dimension], mode="lines",
                             name=f"dimension {dimension}"))
fig.update_layout(title="Selected sinusoidal dimensions across token positions", template="plotly_white",
                  xaxis_title="Position", yaxis_title="Encoding value")
save_plotly_figure(fig, "positional_encoding_curves.html")
fig.show()

pos_layer = SinusoidalPositionEncoding(80, 64)
embeddings_demo = tf.random.normal([1, 12, 64], seed=SEED)
combined_demo = pos_layer(embeddings_demo)
print("Embeddings:", embeddings_demo.shape, "after adding positions:", combined_demo.shape)
assert combined_demo.shape == embeddings_demo.shape

Embeddings: (1, 12, 64) after adding positions: (1, 12, 64)


## Padding and causal masks

Two masks solve different problems:

- **Padding mask:** a `[PAD]` key is artificial and must not receive attention. This is used for source keys and decoder target keys.
- **Causal look-ahead mask:** a decoder position `t` must not attend to target positions greater than `t`. This stops future target information leaking into next-token prediction during training.

A value of `True` in the masks below means “mask this connection.” The attention implementation replaces masked scores with a very negative value before softmax. Valid attention probabilities should sum approximately to one over unmasked keys.

In [14]:
def create_padding_mask(token_ids, pad_id=PAD_ID):
    token_ids = tf.convert_to_tensor(token_ids)
    return tf.equal(token_ids[:, tf.newaxis, tf.newaxis, :], pad_id)


def create_look_ahead_mask(sequence_length):
    lower_triangle = tf.linalg.band_part(tf.ones([sequence_length, sequence_length], dtype=tf.int32), -1, 0)
    mask = 1 - lower_triangle
    return tf.cast(mask[tf.newaxis, tf.newaxis, :, :], tf.bool)


source_example = tf.constant([[5, 8, 3, PAD_ID, PAD_ID]], dtype=tf.int32)
target_example = tf.constant([[BOS_ID, 7, 9, EOS_ID, PAD_ID]], dtype=tf.int32)
source_mask_demo = create_padding_mask(source_example)
target_padding_demo = create_padding_mask(target_example)
causal_demo = create_look_ahead_mask(tf.shape(target_example)[1])
combined_demo_mask = tf.logical_or(target_padding_demo, causal_demo)

fig = make_subplots(rows=1, cols=3, subplot_titles=("Source padding mask", "Target padding mask", "Causal mask"))
fig.add_trace(go.Heatmap(z=source_mask_demo.numpy()[0, 0].astype(int), colorscale=[[0, "#d1fae5"], [1, "#ef4444"]],
                         zmin=0, zmax=1, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=target_padding_demo.numpy()[0, 0].astype(int), colorscale=[[0, "#d1fae5"], [1, "#ef4444"]],
                         zmin=0, zmax=1, showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=causal_demo.numpy()[0, 0].astype(int), colorscale=[[0, "#d1fae5"], [1, "#ef4444"]],
                         zmin=0, zmax=1, showscale=False), row=1, col=3)
fig.update_layout(title="Mask values: green = allowed, red = masked", template="plotly_white", height=420)
save_plotly_figure(fig, "padding_and_causal_masks.html")
fig.show()

scores_demo = tf.zeros([1, 1, 5, 5], dtype=tf.float32)
_, weights_unmasked, _, _ = scaled_dot_product_attention(tf.ones([1, 1, 5, 4]), tf.ones([1, 1, 5, 4]),
                                                           tf.ones([1, 1, 5, 4]))
_, weights_masked, _, _ = scaled_dot_product_attention(tf.ones([1, 1, 5, 4]), tf.ones([1, 1, 5, 4]),
                                                         tf.ones([1, 1, 5, 4]), mask=causal_demo)
print("Causal mask shape:", causal_demo.shape)
print("Masked attention probabilities (upper triangle should be near zero):\n", np.round(weights_masked.numpy()[0, 0], 4))
assert np.allclose(np.triu(weights_masked.numpy()[0, 0], k=1), 0.0, atol=1e-6)

Causal mask shape: (1, 1, 5, 5)
Masked attention probabilities (upper triangle should be near zero):
 [[1.     0.     0.     0.     0.    ]
 [0.5    0.5    0.     0.     0.    ]
 [0.3333 0.3333 0.3333 0.     0.    ]
 [0.25   0.25   0.25   0.25   0.    ]
 [0.2    0.2    0.2    0.2    0.2   ]]


## Feed-forward, residual connections, and normalization

The position-wise feed-forward network is

\[
FFN(x)=W_2\,\mathrm{ReLU}(W_1x+b_1)+b_2.
\]

The same two dense transformations are applied independently at each sequence position. The feature dimension expands from `d_model` to `d_ff`, then contracts back to `d_model`. The original Transformer block uses post-normalization: `LayerNorm(x + Sublayer(x))`. A residual path gives the model a direct route to preserve input information and helps gradients travel through depth. Layer normalization operates over features for each position; because its learned scale and offset are trainable, its outputs are not guaranteed to have exactly zero mean and unit variance after training.

In [15]:
class PositionWiseFeedForward(tf.keras.layers.Layer):
    def __init__(self, d_model, d_ff, **kwargs):
        super().__init__(**kwargs)
        self.expand = tf.keras.layers.Dense(d_ff, activation="relu", name="expand")
        self.contract = tf.keras.layers.Dense(d_model, name="contract")

    def call(self, inputs):
        return self.contract(self.expand(inputs))


ffn_demo = PositionWiseFeedForward(64, 128)
ffn_input = tf.random.normal([3, 7, 64], seed=SEED)
ffn_output = ffn_demo(ffn_input)
expected_params = 64 * 128 + 128 + 128 * 64 + 64
print("Input shape:", ffn_input.shape, "output shape:", ffn_output.shape)
print("Expected FFN parameters:", expected_params)
print("Actual FFN parameters:", ffn_demo.count_params())
assert ffn_output.shape == ffn_input.shape
assert ffn_demo.count_params() == expected_params

sample_activations = tf.random.normal([256, 64], mean=0.7, stddev=2.2, seed=SEED)
normalized_activations = tf.keras.layers.LayerNormalization(epsilon=1e-6)(sample_activations)
stat_frame = pd.DataFrame({"before": sample_activations.numpy().ravel(), "after": normalized_activations.numpy().ravel()})
fig = px.histogram(stat_frame.melt(var_name="stage", value_name="activation"), x="activation", color="stage",
                   barmode="overlay", opacity=0.65, nbins=60, title="Illustration: activation distributions before and after layer normalization")
fig.update_layout(template="plotly_white")
save_plotly_figure(fig, "layer_norm_activation_distribution.html")
fig.show()
print("Mean of normalized features over one example:", float(tf.reduce_mean(normalized_activations[0]).numpy()))
print("Std. dev. over one example:", float(tf.math.reduce_std(normalized_activations[0]).numpy()))

Input shape: (3, 7, 64) output shape: (3, 7, 64)
Expected FFN parameters: 16576
Actual FFN parameters: 16576


Mean of normalized features over one example: 7.450580596923828e-09
Std. dev. over one example: 0.9999998211860657


## Build the encoder, decoder, and full model

The custom implementation below uses only ordinary Keras layers and TensorFlow operations. It does **not** call `keras.layers.MultiHeadAttention` or import an encoder–decoder Transformer implementation. The forward pass remains differentiable, allowing `tf.GradientTape` to train every learned projection and embedding.

The encoder layer contains self-attention, residual-plus-normalization, feed-forward, and residual-plus-normalization. The decoder layer adds masked self-attention first, then cross-attention that queries with decoder states and reads keys/values from the encoder memory, and then the feed-forward network. Both stacks expose attention weights for inspection.

In [16]:
class EncoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.self_attention = MultiHeadAttention(d_model, num_heads, name="self_attention")
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, name="feed_forward")
        self.norm_attention = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="norm_attention")
        self.norm_ffn = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="norm_ffn")
        self.dropout_attention = tf.keras.layers.Dropout(dropout_rate)
        self.dropout_ffn = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, padding_mask=None, training=False):
        attended, weights = self.self_attention(inputs, inputs, inputs, mask=padding_mask, training=training)
        attended = self.dropout_attention(attended, training=training)
        after_attention = self.norm_attention(inputs + attended)
        transformed = self.feed_forward(after_attention)
        transformed = self.dropout_ffn(transformed, training=training)
        outputs = self.norm_ffn(after_attention + transformed)
        return outputs, weights


class DecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, d_ff, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.self_attention = MultiHeadAttention(d_model, num_heads, name="masked_self_attention")
        self.cross_attention = MultiHeadAttention(d_model, num_heads, name="cross_attention")
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, name="feed_forward")
        self.norm_self = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="norm_self")
        self.norm_cross = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="norm_cross")
        self.norm_ffn = tf.keras.layers.LayerNormalization(epsilon=1e-6, name="norm_ffn")
        self.dropout_self = tf.keras.layers.Dropout(dropout_rate)
        self.dropout_cross = tf.keras.layers.Dropout(dropout_rate)
        self.dropout_ffn = tf.keras.layers.Dropout(dropout_rate)

    def call(self, inputs, encoder_memory, self_mask=None, cross_mask=None, training=False):
        self_output, self_weights = self.self_attention(inputs, inputs, inputs, mask=self_mask, training=training)
        self_output = self.dropout_self(self_output, training=training)
        after_self = self.norm_self(inputs + self_output)

        cross_output, cross_weights = self.cross_attention(after_self, encoder_memory, encoder_memory,
                                                            mask=cross_mask, training=training)
        cross_output = self.dropout_cross(cross_output, training=training)
        after_cross = self.norm_cross(after_self + cross_output)

        transformed = self.feed_forward(after_cross)
        transformed = self.dropout_ffn(transformed, training=training)
        outputs = self.norm_ffn(after_cross + transformed)
        return outputs, self_weights, cross_weights


class TransformerEncoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_length, dropout_rate=0.1,
                 use_positional_encoding=True, **kwargs):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, name="token_embedding")
        self.position = SinusoidalPositionEncoding(max_length, d_model, name="sinusoidal_position")
        self.use_positional_encoding = bool(use_positional_encoding)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.blocks = [EncoderLayer(d_model, num_heads, d_ff, dropout_rate, name=f"encoder_block_{i+1}")
                       for i in range(num_layers)]

    def call(self, token_ids, padding_mask=None, training=False):
        x = self.embedding(token_ids)
        x = x * tf.math.sqrt(tf.cast(self.d_model, x.dtype))
        if self.use_positional_encoding:
            x = self.position(x)
        x = self.dropout(x, training=training)
        attention_weights = []
        representations = [x]
        for block in self.blocks:
            x, weights = block(x, padding_mask=padding_mask, training=training)
            attention_weights.append(weights)
            representations.append(x)
        return x, attention_weights, representations


class TransformerDecoder(tf.keras.layers.Layer):
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, max_length, dropout_rate=0.1,
                 use_positional_encoding=True, **kwargs):
        super().__init__(**kwargs)
        self.d_model = int(d_model)
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model, name="token_embedding")
        self.position = SinusoidalPositionEncoding(max_length, d_model, name="sinusoidal_position")
        self.use_positional_encoding = bool(use_positional_encoding)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.blocks = [DecoderLayer(d_model, num_heads, d_ff, dropout_rate, name=f"decoder_block_{i+1}")
                       for i in range(num_layers)]

    def call(self, token_ids, encoder_memory, self_mask=None, cross_mask=None, training=False):
        x = self.embedding(token_ids)
        x = x * tf.math.sqrt(tf.cast(self.d_model, x.dtype))
        if self.use_positional_encoding:
            x = self.position(x)
        x = self.dropout(x, training=training)
        self_weights, cross_weights, representations = [], [], [x]
        for block in self.blocks:
            x, self_attention, cross_attention = block(x, encoder_memory, self_mask=self_mask,
                                                       cross_mask=cross_mask, training=training)
            self_weights.append(self_attention)
            cross_weights.append(cross_attention)
            representations.append(x)
        return x, self_weights, cross_weights, representations


class Transformer(tf.keras.Model):
    def __init__(self, source_vocab_size, target_vocab_size, d_model=64, num_heads=4,
                 num_encoder_layers=1, num_decoder_layers=1, d_ff=128, dropout_rate=0.1,
                 source_max_length=48, target_max_length=48, use_positional_encoding=True, **kwargs):
        super().__init__(**kwargs)
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.source_vocab_size = int(source_vocab_size)
        self.target_vocab_size = int(target_vocab_size)
        self.d_model = int(d_model)
        self.num_heads = int(num_heads)
        self.num_encoder_layers = int(num_encoder_layers)
        self.num_decoder_layers = int(num_decoder_layers)
        self.d_ff = int(d_ff)
        self.dropout_rate = float(dropout_rate)
        self.source_max_length = int(source_max_length)
        self.target_max_length = int(target_max_length)
        self.use_positional_encoding = bool(use_positional_encoding)
        self.encoder = TransformerEncoder(source_vocab_size, d_model, num_heads, d_ff, num_encoder_layers,
                                         source_max_length, dropout_rate, use_positional_encoding, name="encoder")
        self.decoder = TransformerDecoder(target_vocab_size, d_model, num_heads, d_ff, num_decoder_layers,
                                         target_max_length, dropout_rate, use_positional_encoding, name="decoder")
        self.output_projection = tf.keras.layers.Dense(target_vocab_size, name="vocabulary_projection")

    def encode(self, source_ids, training=False):
        source_ids = tf.convert_to_tensor(source_ids, dtype=tf.int32)
        tf.debugging.assert_greater_equal(source_ids, 0, message="Source token IDs must be non-negative")
        tf.debugging.assert_less(source_ids, self.source_vocab_size, message="Source token ID exceeds source vocabulary")
        source_padding_mask = create_padding_mask(source_ids, PAD_ID)
        memory, encoder_attention, encoder_representations = self.encoder(
            source_ids, padding_mask=source_padding_mask, training=training
        )
        return memory, source_padding_mask, encoder_attention, encoder_representations

    def decode(self, target_ids, memory, source_padding_mask, training=False):
        target_ids = tf.convert_to_tensor(target_ids, dtype=tf.int32)
        tf.debugging.assert_greater_equal(target_ids, 0, message="Target token IDs must be non-negative")
        tf.debugging.assert_less(target_ids, self.target_vocab_size, message="Target token ID exceeds target vocabulary")
        target_padding_mask = create_padding_mask(target_ids, PAD_ID)
        causal_mask = create_look_ahead_mask(tf.shape(target_ids)[1])
        self_mask = tf.logical_or(target_padding_mask, causal_mask)
        decoder_output, decoder_self_attention, cross_attention, decoder_representations = self.decoder(
            target_ids, memory, self_mask=self_mask, cross_mask=source_padding_mask, training=training
        )
        logits = self.output_projection(decoder_output)
        return logits, decoder_self_attention, cross_attention, decoder_representations

    def call(self, inputs, training=False, return_attention=False):
        source_ids, target_ids = inputs
        tf.debugging.assert_equal(tf.shape(source_ids)[0], tf.shape(target_ids)[0],
                                  message="Source and target batch sizes must match")
        memory, source_padding_mask, encoder_attention, encoder_representations = self.encode(source_ids, training=training)
        logits, decoder_self_attention, cross_attention, decoder_representations = self.decode(
            target_ids, memory, source_padding_mask, training=training
        )
        if return_attention:
            return logits, {
                "encoder_self_attention": encoder_attention,
                "decoder_self_attention": decoder_self_attention,
                "cross_attention": cross_attention,
                "encoder_representations": encoder_representations,
                "decoder_representations": decoder_representations,
            }
        return logits


model = Transformer(
    source_vocab_size=len(source_tokenizer.token_to_id),
    target_vocab_size=len(target_tokenizer.token_to_id),
    d_model=CONFIG["d_model"], num_heads=CONFIG["num_heads"],
    num_encoder_layers=CONFIG["num_encoder_layers"], num_decoder_layers=CONFIG["num_decoder_layers"],
    d_ff=CONFIG["d_ff"], dropout_rate=CONFIG["dropout"],
    source_max_length=SRC_MAX_LEN, target_max_length=TGT_MAX_LEN,
)

warmup_source = tf.constant(train_src[:2], dtype=tf.int32)
warmup_target = tf.constant(train_tgt_in[:2], dtype=tf.int32)
warmup_logits = model((warmup_source, warmup_target), training=False)
print("Model logits shape:", warmup_logits.shape)
print("Expected shape:", (2, train_tgt_in.shape[1], len(target_tokenizer.token_to_id)))
assert warmup_logits.shape == (2, train_tgt_in.shape[1], len(target_tokenizer.token_to_id))
assert np.isfinite(warmup_logits.numpy()).all()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'masked_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.



Model logits shape: (2, 15, 202)
Expected shape: (2, 15, 202)


### Architecture summary table

The table below uses the actual padded shapes chosen for this run. Parameter counts are filled from the instantiated model after its first forward pass, rather than copied from a generic diagram.

In [17]:
B_EXAMPLE = 2
S_EXAMPLE = int(train_src.shape[1])
T_EXAMPLE = int(train_tgt_in.shape[1])
D_EXAMPLE = CONFIG["d_model"]
H_EXAMPLE = CONFIG["num_heads"]
DH_EXAMPLE = D_EXAMPLE // H_EXAMPLE
V_SOURCE = len(source_tokenizer.token_to_id)
V_TARGET = len(target_tokenizer.token_to_id)

architecture_table = pd.DataFrame([
    ["Source IDs", f"[{B_EXAMPLE}, {S_EXAMPLE}]", f"[{B_EXAMPLE}, {S_EXAMPLE}]", "Integer lookup input", "Represent source tokens", "No"],
    ["Source embedding + position", f"[{B_EXAMPLE}, {S_EXAMPLE}]", f"[{B_EXAMPLE}, {S_EXAMPLE}, {D_EXAMPLE}]", "Embedding lookup + positional addition", "Create token representations", "Yes: embedding"],
    ["Encoder attention", f"[{B_EXAMPLE}, {S_EXAMPLE}, {D_EXAMPLE}]", f"[{B_EXAMPLE}, {S_EXAMPLE}, {D_EXAMPLE}]", "Multi-head self-attention", "Contextualize source tokens", "Yes: Q/K/V/output projections"],
    ["Encoder memory", f"[{B_EXAMPLE}, {S_EXAMPLE}, {D_EXAMPLE}]", f"[{B_EXAMPLE}, {S_EXAMPLE}, {D_EXAMPLE}]", "Stacked encoder blocks", "Supply keys and values to decoder", "Yes: encoder blocks"],
    ["Target prefix", f"[{B_EXAMPLE}, {T_EXAMPLE}]", f"[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}]", "Target embedding + positions", "Represent generated prefix", "Yes: embedding"],
    ["Decoder self-attention", f"[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}]", f"[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}]", "Masked multi-head attention", "Use only current/past target tokens", "Yes: Q/K/V/output projections"],
    ["Cross-attention", f"Q=[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}], K/V=[{B_EXAMPLE}, {S_EXAMPLE}, {D_EXAMPLE}]", f"[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}]", "Decoder queries attend to encoder memory", "Connect output to source", "Yes: Q/K/V/output projections"],
    ["Feed-forward", f"[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}]", f"[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}]", f"Dense {D_EXAMPLE} → {CONFIG['d_ff']} → {D_EXAMPLE}", "Transform each position independently", "Yes: dense weights and biases"],
    ["Vocabulary logits", f"[{B_EXAMPLE}, {T_EXAMPLE}, {D_EXAMPLE}]", f"[{B_EXAMPLE}, {T_EXAMPLE}, {V_TARGET}]", "Linear projection", "Score each target token", "Yes: output projection"],
], columns=["Component", "Input tensor shape", "Output tensor shape", "Operation", "Purpose", "Learnable parameters"])
display(architecture_table)

,Component,Input tensor shape,Output tensor shape,Operation,Purpose,Learnable parameters
0,Source IDs,"[2, 15]","[2, 15]",Integer lookup input,Represent source tokens,No
1,Source embedding + position,"[2, 15]","[2, 15, 64]",Embedding lookup + positional addition,Create token representations,Yes: embedding
2,Encoder attention,"[2, 15, 64]","[2, 15, 64]",Multi-head self-attention,Contextualize source tokens,Yes: Q/K/V/output projections
3,Encoder memory,"[2, 15, 64]","[2, 15, 64]",Stacked encoder blocks,Supply keys and values to decoder,Yes: encoder blocks
4,Target prefix,"[2, 15]","[2, 15, 64]",Target embedding + positions,Represent generated prefix,Yes: embedding
5,Decoder self-attention,"[2, 15, 64]","[2, 15, 64]",Masked multi-head attention,Use only current/past target tokens,Yes: Q/K/V/output projections
6,Cross-attention,"Q=[2, 15, 64], K/V=[2, 15, 64]","[2, 15, 64]",Decoder queries attend to encoder memory,Connect output to source,Yes: Q/K/V/output projections
7,Feed-forward,"[2, 15, 64]","[2, 15, 64]",Dense 64 → 128 → 64,Transform each position independently,Yes: dense weights and biases
8,Vocabulary logits,"[2, 15, 64]","[2, 15, 202]",Linear projection,Score each target token,Yes: output projection


## Teacher forcing and masked loss

The target sequence is shifted by one position. Given the target `[BOS] the answer is 7 [EOS]`, the decoder input is `[BOS] the answer is 7` while the expected labels are `the answer is 7 [EOS]`. During training, the decoder sees the correct prefix at every position; causal masking ensures it cannot inspect future positions. This technique is called **teacher forcing**.

At inference, the correct target prefix is not available. The model must generate a token, append it to its own prefix, and predict again. The notebook later uses greedy autoregressive decoding for this reason.

The loss uses raw vocabulary logits with sparse categorical cross-entropy. Padding labels are ignored when averaging loss and computing token accuracy.

In [18]:
def masked_sparse_cross_entropy(labels, logits, pad_id=PAD_ID):
    labels = tf.cast(labels, tf.int32)
    per_token_loss = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=logits)
    valid = tf.cast(tf.not_equal(labels, pad_id), per_token_loss.dtype)
    return tf.reduce_sum(per_token_loss * valid) / tf.maximum(tf.reduce_sum(valid), 1.0)


def masked_token_accuracy(labels, logits, pad_id=PAD_ID):
    labels = tf.cast(labels, tf.int32)
    predictions = tf.argmax(logits, axis=-1, output_type=tf.int32)
    valid = tf.not_equal(labels, pad_id)
    correct = tf.logical_and(tf.equal(labels, predictions), valid)
    return tf.reduce_sum(tf.cast(correct, tf.float32)) / tf.maximum(tf.reduce_sum(tf.cast(valid, tf.float32)), 1.0)


shift_example = target_tokenizer.encode("sum is 7 .", add_bos=True, add_eos=True)
print("Full target:", [target_tokenizer.id_to_token[i] for i in shift_example])
print("Decoder input:", [target_tokenizer.id_to_token[i] for i in shift_example[:-1]])
print("Expected labels:", [target_tokenizer.id_to_token[i] for i in shift_example[1:]])

fake_logits = tf.zeros([1, len(shift_example) - 1, len(target_tokenizer.token_to_id)], dtype=tf.float32)
labels_demo = tf.constant([shift_example[1:]], dtype=tf.int32)
print("Masked loss on uniform logits:", float(masked_sparse_cross_entropy(labels_demo, fake_logits).numpy()))
print("Token accuracy on uniform logits:", float(masked_token_accuracy(labels_demo, fake_logits).numpy()))
assert labels_demo.shape[1] == fake_logits.shape[1]

Full target: ['[BOS]', 'sum', 'is', '7', '.', '[EOS]']
Decoder input: ['[BOS]', 'sum', 'is', '7', '.']
Expected labels: ['sum', 'is', '7', '.', '[EOS]']
Masked loss on uniform logits: 5.308267593383789
Token accuracy on uniform logits: 0.0


## Training

Training is from scratch. The learning rate follows a Transformer-style warm-up and inverse-square-root schedule. The custom training step clips the global gradient norm, applies the optimizer, and tracks padding-aware loss, token accuracy, and gradient norm. Validation is run without dropout; the best validation checkpoint is retained and restored before evaluation.

The default settings are a practical starting point, not a promise of a particular score. If CPU training is slow, reduce `examples_per_task`, use fewer epochs, lower `d_model`, or start with a tiny batch-overfit test. If loss stalls, inspect the sample generations, masks, target shift, learning rate, and unknown-token rates before simply increasing model size.

In [19]:
def make_tf_dataset(source_ids, target_inputs, target_labels, batch_size, shuffle=False, seed=SEED):
    data = tf.data.Dataset.from_tensor_slices((source_ids, target_inputs, target_labels))
    if shuffle:
        data = data.shuffle(buffer_size=len(source_ids), seed=seed, reshuffle_each_iteration=True)
    data = data.batch(batch_size, drop_remainder=False)
    options = tf.data.Options()
    options.experimental_deterministic = True
    data = data.with_options(options)
    return data.prefetch(tf.data.AUTOTUNE)


train_dataset = make_tf_dataset(train_src, train_tgt_in, train_labels, CONFIG["batch_size"], shuffle=True)
validation_dataset = make_tf_dataset(val_src, val_tgt_in, val_labels, CONFIG["batch_size"], shuffle=False)

class TransformerLearningRate(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, d_model, warmup_steps=300):
        super().__init__()
        self.d_model = float(d_model)
        self.warmup_steps = float(warmup_steps)

    def __call__(self, step):
        step = tf.cast(step, tf.float32) + 1.0
        inverse_sqrt = tf.math.rsqrt(step)
        warmup = step * (self.warmup_steps ** -1.5)
        return tf.math.rsqrt(self.d_model) * tf.minimum(inverse_sqrt, warmup)


learning_rate_schedule = TransformerLearningRate(CONFIG["d_model"], CONFIG["warmup_steps"])
optimizer = tf.keras.optimizers.Adam(learning_rate_schedule, beta_1=0.9, beta_2=0.98, epsilon=1e-9)

@tf.function(reduce_retracing=True)
def train_step(source_batch, target_input_batch, target_label_batch):
    with tf.GradientTape() as tape:
        predictions = model((source_batch, target_input_batch), training=True)
        loss_value = masked_sparse_cross_entropy(target_label_batch, predictions)
    gradients = tape.gradient(loss_value, model.trainable_variables)
    gradient_pairs = [(gradient, variable) for gradient, variable in zip(gradients, model.trainable_variables) if gradient is not None]
    gradient_values = [pair[0] for pair in gradient_pairs]
    gradient_norm = tf.linalg.global_norm(gradient_values)
    clipped_gradients, _ = tf.clip_by_global_norm(gradient_values, CONFIG["gradient_clip_norm"])
    optimizer.apply_gradients(zip(clipped_gradients, [pair[1] for pair in gradient_pairs]))
    accuracy_value = masked_token_accuracy(target_label_batch, predictions)
    return loss_value, accuracy_value, gradient_norm


def validation_step(source_batch, target_input_batch, target_label_batch):
    predictions = model((source_batch, target_input_batch), training=False)
    loss_value = masked_sparse_cross_entropy(target_label_batch, predictions)
    accuracy_value = masked_token_accuracy(target_label_batch, predictions)
    return loss_value, accuracy_value

In [20]:
EPOCHS = CONFIG["epochs"]
PATIENCE = CONFIG["patience"]
checkpoint_path = OUTPUT_DIRS["checkpoints"] / "best_transformer.weights.h5"
history_records = []
best_validation_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()
    train_losses, train_accuracies, epoch_gradient_norms = [], [], []
    for source_batch, target_input_batch, target_label_batch in train_dataset:
        loss_value, accuracy_value, gradient_norm = train_step(source_batch, target_input_batch, target_label_batch)
        train_losses.append(float(loss_value.numpy()))
        train_accuracies.append(float(accuracy_value.numpy()))
        epoch_gradient_norms.append(float(gradient_norm.numpy()))

    validation_losses, validation_accuracies = [], []
    for source_batch, target_input_batch, target_label_batch in validation_dataset:
        loss_value, accuracy_value = validation_step(source_batch, target_input_batch, target_label_batch)
        validation_losses.append(float(loss_value.numpy()))
        validation_accuracies.append(float(accuracy_value.numpy()))

    epoch_seconds = time.perf_counter() - epoch_start
    current_lr = float(optimizer.learning_rate.numpy()) if hasattr(optimizer.learning_rate, "numpy") else float(optimizer.learning_rate)
    record = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        "validation_loss": float(np.mean(validation_losses)),
        "train_token_accuracy": float(np.mean(train_accuracies)),
        "validation_token_accuracy": float(np.mean(validation_accuracies)),
        "mean_gradient_norm": float(np.mean(epoch_gradient_norms)),
        "learning_rate": current_lr,
        "epoch_seconds": epoch_seconds,
    }
    history_records.append(record)
    print(f"Epoch {epoch:02d}/{EPOCHS} | train loss {record['train_loss']:.4f} | "
          f"val loss {record['validation_loss']:.4f} | train token acc {record['train_token_accuracy']:.3f} | "
          f"val token acc {record['validation_token_accuracy']:.3f} | {epoch_seconds:.1f}s")

    if record["validation_loss"] < best_validation_loss:
        best_validation_loss = record["validation_loss"]
        epochs_without_improvement = 0
        model.save_weights(checkpoint_path)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            print("Early stopping: validation loss did not improve within the patience window.")
            break

history_df = pd.DataFrame(history_records)
history_df.to_csv(OUTPUT_DIRS["metrics"] / "training_history.csv", index=False)
model.load_weights(checkpoint_path)
with open(OUTPUT_DIRS["logs"] / "training_summary.json", "w", encoding="utf-8") as file:
    json.dump({"best_validation_loss": best_validation_loss, "epochs_completed": len(history_df),
               "config": CONFIG, "tensorflow_version": tf.__version__}, file, indent=2)

display(history_df)

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'masked_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.



Epoch 01/6 | train loss 1.9683 | val loss 0.4273 | train token acc 0.605 | val token acc 0.878 | 10.6s
Epoch 02/6 | train loss 0.3339 | val loss 0.2416 | train token acc 0.906 | val token acc 0.932 | 3.2s
Epoch 03/6 | train loss 0.2305 | val loss 0.1797 | train token acc 0.935 | val token acc 0.948 | 3.2s
Epoch 04/6 | train loss 0.1684 | val loss 0.1224 | train token acc 0.951 | val token acc 0.964 | 3.3s
Epoch 05/6 | train loss 0.1409 | val loss 0.1068 | train token acc 0.958 | val token acc 0.967 | 3.2s
Epoch 06/6 | train loss 0.1189 | val loss 0.1064 | train token acc 0.963 | val token acc 0.968 | 3.2s


,epoch,train_loss,validation_loss,train_token_accuracy,validation_token_accuracy,mean_gradient_norm,learning_rate,epoch_seconds
0,1,1.968335,0.427316,0.605275,0.878069,1.007877,0.003151,10.631109
1,2,0.333872,0.241636,0.905890,0.932459,0.537239,0.006279,3.184573
2,3,0.230524,0.179724,0.934907,0.947539,0.547572,0.006322,3.212928
3,4,0.168428,0.122393,0.950699,0.963685,0.510795,0.005476,3.263523
4,5,0.140900,0.106810,0.957698,0.967433,0.452934,0.004899,3.242818
5,6,0.118894,0.106366,0.963061,0.968212,0.453990,0.004473,3.215938


In [21]:
fig = make_subplots(rows=2, cols=2, subplot_titles=("Loss", "Token accuracy", "Learning rate", "Gradient norm"))
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.train_loss, mode="lines+markers", name="Train loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.validation_loss, mode="lines+markers", name="Validation loss"), row=1, col=1)
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.train_token_accuracy, mode="lines+markers", name="Train token accuracy"), row=1, col=2)
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.validation_token_accuracy, mode="lines+markers", name="Validation token accuracy"), row=1, col=2)
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.learning_rate, mode="lines+markers", name="Learning rate"), row=2, col=1)
fig.add_trace(go.Scatter(x=history_df.epoch, y=history_df.mean_gradient_norm, mode="lines+markers", name="Gradient norm"), row=2, col=2)
fig.update_layout(title="Training dashboard", template="plotly_white", height=760)
save_plotly_figure(fig, "training_dashboard.html")
fig.show()

fig = px.bar(history_df, x="epoch", y="epoch_seconds", color="epoch_seconds", color_continuous_scale="Blues",
             title="Training time per epoch", labels={"epoch_seconds": "Seconds"})
fig.update_layout(template="plotly_white")
save_plotly_figure(fig, "epoch_training_time.html")
fig.show()

history_df["generalization_gap"] = history_df["validation_loss"] - history_df["train_loss"]
fig = px.line(history_df, x="epoch", y="generalization_gap", markers=True,
              title="Generalization gap: validation loss minus training loss")
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(template="plotly_white", yaxis_title="Validation loss − training loss")
save_plotly_figure(fig, "generalization_gap.html")
fig.show()

### Reading the training curves

- **Underfitting:** both training and validation loss remain high, often with limited progress.
- **Overfitting:** training loss continues to fall while validation loss rises or the gap widens.
- **Convergence:** training and validation curves stabilize at useful values.
- **Unstable training:** loss or gradient norms spike, become non-finite, or vary wildly between steps.

A smaller validation loss is not enough by itself. Inspect generated outputs and per-task accuracy; a model may do well on frequent easy templates while failing a less common task. If the model repeats tokens, verify the target shift, EOS handling, causal mask, PAD masking, learning rate, and whether the model has trained long enough.

## Autoregressive evaluation

Greedy decoding starts with `[BOS]`, encodes the source once, and repeatedly chooses the highest-logit next token from the final decoder position. It stops when `[EOS]` is produced or the output limit is reached. This uses the model's own generated prefix, not the ground-truth target tokens.

Greedy decoding is fast and easy to inspect but is not guaranteed to find the globally highest-probability sequence. A locally best next token can lead to a worse overall continuation than another token would have produced.

In [22]:
def greedy_decode_batch(model, source_batch, max_new_tokens):
    source_batch = tf.convert_to_tensor(source_batch, dtype=tf.int32)
    memory, source_padding_mask, _, _ = model.encode(source_batch, training=False)
    batch_size = tf.shape(source_batch)[0]
    generated = tf.fill([batch_size, 1], tf.cast(BOS_ID, tf.int32))
    finished = tf.zeros([batch_size], dtype=tf.bool)

    for _ in range(max_new_tokens):
        logits, _, _, _ = model.decode(generated, memory, source_padding_mask, training=False)
        next_token = tf.argmax(logits[:, -1, :], axis=-1, output_type=tf.int32)
        next_token = tf.where(finished, tf.cast(PAD_ID, tf.int32), next_token)
        generated = tf.concat([generated, next_token[:, tf.newaxis]], axis=1)
        finished = tf.logical_or(finished, tf.equal(next_token, EOS_ID))
        if bool(tf.reduce_all(finished).numpy()):
            break
    return generated


def pad_generated_to_width(generated, width):
    generated = np.asarray(generated, dtype=np.int32)
    if generated.shape[1] < width:
        generated = np.pad(generated, ((0, 0), (0, width - generated.shape[1])), constant_values=PAD_ID)
    return generated[:, :width]


def evaluate_autoregressively(model, frame, source_ids, target_full_ids, batch_size=64, limit=None):
    count = len(frame) if limit is None else min(len(frame), int(limit))
    predictions, ground_truths, latencies = [], [], []
    source_slice = source_ids[:count]
    target_slice = target_full_ids[:count]
    frame_slice = frame.iloc[:count].reset_index(drop=True)
    start_all = time.perf_counter()

    for start in range(0, count, batch_size):
        end = min(start + batch_size, count)
        batch_start = time.perf_counter()
        generated = greedy_decode_batch(model, source_slice[start:end], max_new_tokens=TGT_MAX_LEN - 1)
        latencies.append((time.perf_counter() - batch_start) / max(end - start, 1))
        generated = pad_generated_to_width(generated.numpy(), target_slice[start:end].shape[1])
        predictions.extend([target_tokenizer.decode(row[1:]) for row in generated])
        ground_truths.extend([target_tokenizer.decode(row[1:]) for row in target_slice[start:end]])

    total_seconds = time.perf_counter() - start_all
    generated_matrix = np.zeros_like(target_slice)
    for index, tokens in enumerate(predictions):
        encoded = target_tokenizer.encode(tokens, add_bos=True, add_eos=True)
        generated_matrix[index, :min(len(encoded), generated_matrix.shape[1])] = encoded[:generated_matrix.shape[1]]
    aligned_predictions = pad_generated_to_width(generated_matrix[:, 1:], target_slice[:, 1:].shape[1])
    reference_labels = target_slice[:, 1:]
    valid = reference_labels != PAD_ID
    token_accuracy = float(((aligned_predictions == reference_labels) & valid).sum() / max(valid.sum(), 1))
    exact_matches = np.array([pred == truth for pred, truth in zip(predictions, ground_truths)])

    result = frame_slice.copy()
    result["ground_truth"] = ground_truths
    result["prediction"] = predictions
    result["exact_match"] = exact_matches
    result["generated_tokens"] = [len(tokenize(text)) for text in predictions]
    metrics = {
        "evaluated_examples": count,
        "exact_match_accuracy": float(exact_matches.mean()) if count else float("nan"),
        "generated_token_accuracy": token_accuracy,
        "average_generated_length": float(result.generated_tokens.mean()) if count else float("nan"),
        "total_generation_seconds": total_seconds,
        "average_latency_seconds_per_example": total_seconds / max(count, 1),
    }
    return result, metrics


evaluation_frame, generation_metrics = evaluate_autoregressively(model, test_frame, test_src, test_tgt_full,
                                                                  batch_size=CONFIG["batch_size"], limit=None)
print("Autoregressive test metrics")
print(json.dumps(generation_metrics, indent=2))
display(evaluation_frame[["task_type", "source_text", "ground_truth", "prediction", "exact_match"]].head(20))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'masked_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.



Autoregressive test metrics
{
  "evaluated_examples": 1040,
  "exact_match_accuracy": 0.7653846153846153,
  "generated_token_accuracy": 0.9629401625215465,
  "average_generated_length": 6.809615384615385,
  "total_generation_seconds": 11.666183649999994,
  "average_latency_seconds_per_example": 0.011217484278846147
}


,task_type,source_text,ground_truth,prediction,exact_match
0,describe,describe an item color purple size medium mate...,"the item is purple, medium, flat, and made of ...","the item is purple, medium, flat, and made of ...",True
1,sort,"sort these values 7, 4, 2, 2, 2","sorted sequence is 2, 2, 2, 4, 7.","sorted sequence is 2, 4, 4, 7, 4.",False
2,sum,add 12 and 16,sum is 28.,sum is 30.,False
3,parity,check whether 15 is even,number is odd.,number is odd.,True
4,sort,"sort these values 0, 5, 9, 1, 1","sorted sequence is 0, 1, 1, 5, 9.","sorted sequence is 0, 1, 5, 9, 9.",False
5,minimum,"what is the minimum of 8, 9, 6, 3",smallest value is 3.,smallest value is 3.,True
6,sum,add 4 and 6,sum is 10.,sum is 9.,False
7,double,multiply 23 by two,double is 46.,double is 46.,True
8,reverse,"reverse these values 2, 4, 6","reversed sequence is 6, 4, 2.","reversed sequence is 6, 4, 2.",True
9,reverse,"write 0, 7, 0, 1, 7 in reverse order","reversed sequence is 7, 1, 0, 7, 0.","reversed sequence is 7, 1, 7, 0, 0.",False


In [23]:
test_teacher_forced_logits = model((tf.constant(test_src), tf.constant(test_tgt_in)), training=False)
test_teacher_forced_loss = float(masked_sparse_cross_entropy(tf.constant(test_labels), test_teacher_forced_logits).numpy())
test_teacher_forced_accuracy = float(masked_token_accuracy(tf.constant(test_labels), test_teacher_forced_logits).numpy())

per_task_metrics = (evaluation_frame.groupby("task_type", as_index=False)
                    .agg(examples=("exact_match", "size"), exact_match_accuracy=("exact_match", "mean"),
                         average_generated_length=("generated_tokens", "mean")))
final_metrics = {
    **generation_metrics,
    "teacher_forced_test_loss": test_teacher_forced_loss,
    "teacher_forced_test_token_accuracy": test_teacher_forced_accuracy,
    "tensorflow_version": tf.__version__,
    "model_config": CONFIG,
}
with open(OUTPUT_DIRS["metrics"] / "test_metrics.json", "w", encoding="utf-8") as file:
    json.dump(final_metrics, file, indent=2)
evaluation_frame.to_csv(OUTPUT_DIRS["metrics"] / "test_predictions.csv", index=False)
per_task_metrics.to_csv(OUTPUT_DIRS["metrics"] / "per_task_metrics.csv", index=False)

display(per_task_metrics.sort_values("exact_match_accuracy", ascending=False))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'masked_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.



,task_type,examples,exact_match_accuracy,average_generated_length
1,copy,80,1.0000,11.275
2,describe,80,1.0000,14.000
7,maximum,80,1.0000,5.000
6,last,80,1.0000,5.000
5,first,80,1.0000,5.000
8,minimum,80,1.0000,5.000
9,parity,80,0.9875,4.000
4,double,80,0.9250,4.000
0,compare,80,0.8875,5.000
11,sort,80,0.5125,11.150


In [24]:
fig = px.bar(per_task_metrics.sort_values("exact_match_accuracy"), x="exact_match_accuracy", y="task_type", orientation="h",
             color="exact_match_accuracy", color_continuous_scale="Viridis", range_x=[0, 1],
             title="Autoregressive exact-match accuracy by task", labels={"exact_match_accuracy": "Exact-match accuracy", "task_type": "Task"})
fig.update_layout(template="plotly_white", height=650)
save_plotly_figure(fig, "task_accuracy.html")
fig.show()

comparison = evaluation_frame.head(40).copy()
comparison["status"] = np.where(comparison["exact_match"], "Exact match", "Mismatch")
fig = px.scatter(comparison, x="task_type", y="generated_tokens", color="status", hover_data=["source_text", "ground_truth", "prediction"],
                 title="Generated output length and prediction status on sample test cases")
fig.update_layout(template="plotly_white", xaxis_tickangle=-35)
save_plotly_figure(fig, "prediction_examples.html")
fig.show()

error_frame = evaluation_frame.loc[~evaluation_frame.exact_match].copy()
if len(error_frame):
    error_counts = error_frame.task_type.value_counts().rename_axis("task_type").reset_index(name="errors")
    fig = px.bar(error_counts, x="task_type", y="errors", color="errors", color_continuous_scale="OrRd",
                 title="Error counts by task on the held-out test set")
    fig.update_layout(template="plotly_white", xaxis_tickangle=-35)
    save_plotly_figure(fig, "error_analysis.html")
    fig.show()
else:
    print("No exact-match errors on the evaluated set; error-count plot is not applicable.")

## Visual analytics

This section inspects **actual intermediate values from the trained model**. The attention maps are generated from the model's own forward pass for a held-out example; they are not hand-drawn attention weights. The axes show token strings so the plots can be interpreted as relationships between real source/target positions.

In [25]:
selected_row = evaluation_frame.iloc[0]
selected_source_ids = tf.constant(test_src[[0]], dtype=tf.int32)
selected_target_prefix = tf.constant(test_tgt_in[[0]], dtype=tf.int32)
selected_logits, selected_details = model((selected_source_ids, selected_target_prefix), training=False, return_attention=True)

source_tokens_for_plot = [source_tokenizer.id_to_token.get(int(token), UNK_TOKEN) for token in test_src[0] if int(token) != PAD_ID]
target_tokens_for_plot = [target_tokenizer.id_to_token.get(int(token), UNK_TOKEN) for token in test_tgt_in[0] if int(token) != PAD_ID]
print("Source:", selected_row.source_text)
print("Ground truth:", selected_row.ground_truth)
print("Generated:", selected_row.prediction)
print("Logits shape:", selected_logits.shape)
print("Encoder attention shape:", selected_details["encoder_self_attention"][0].shape)
print("Decoder self-attention shape:", selected_details["decoder_self_attention"][0].shape)
print("Cross-attention shape:", selected_details["cross_attention"][0].shape)

Source: describe an item color purple size medium material glass shape flat
Ground truth: the item is purple, medium, flat, and made of glass.
Generated: the item is purple, medium, flat, and made of glass.
Logits shape: (1, 15, 202)
Encoder attention shape: (1, 4, 15, 15)
Decoder self-attention shape: (1, 4, 15, 15)
Cross-attention shape: (1, 4, 15, 15)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'masked_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.



In [26]:
encoder_weights = selected_details["encoder_self_attention"][0].numpy()[0]
decoder_self_weights = selected_details["decoder_self_attention"][0].numpy()[0]
cross_weights = selected_details["cross_attention"][0].numpy()[0]

for head_index, head_matrix in enumerate(encoder_weights):
    valid_length = len(source_tokens_for_plot)
    attention_heatmap(head_matrix[:valid_length, :valid_length], source_tokens_for_plot, source_tokens_for_plot,
                      f"Encoder self-attention probabilities — head {head_index + 1}",
                      f"encoder_attention_head_{head_index + 1}.html", "Viridis", 0, 1)

valid_target_length = len(target_tokens_for_plot)
for head_index, head_matrix in enumerate(decoder_self_weights):
    attention_heatmap(head_matrix[:valid_target_length, :valid_target_length], target_tokens_for_plot, target_tokens_for_plot,
                      f"Decoder masked self-attention probabilities — head {head_index + 1}",
                      f"decoder_self_attention_head_{head_index + 1}.html", "Viridis", 0, 1)

for head_index, head_matrix in enumerate(cross_weights):
    attention_heatmap(head_matrix[:valid_target_length, :len(source_tokens_for_plot)], source_tokens_for_plot, target_tokens_for_plot,
                      f"Encoder–decoder cross-attention probabilities — head {head_index + 1}",
                      f"cross_attention_head_{head_index + 1}.html", "Viridis", 0, 1)

In [27]:
encoder_representations = selected_details["encoder_representations"][-1].numpy()[0]
decoder_representations = selected_details["decoder_representations"][-1].numpy()[0]

fig = px.imshow(encoder_representations[:len(source_tokens_for_plot)].T, aspect="auto", origin="lower",
                x=source_tokens_for_plot, color_continuous_scale="RdBu_r",
                title="Encoder memory representations for the selected source", labels={"x": "Source token", "y": "Feature dimension", "color": "Activation"})
fig.update_layout(template="plotly_white", height=600)
save_plotly_figure(fig, "encoder_representation_heatmap.html")
fig.show()

fig = px.imshow(decoder_representations[:len(target_tokens_for_plot)].T, aspect="auto", origin="lower",
                x=target_tokens_for_plot, color_continuous_scale="RdBu_r",
                title="Decoder representations for the selected target prefix", labels={"x": "Target prefix token", "y": "Feature dimension", "color": "Activation"})
fig.update_layout(template="plotly_white", height=600)
save_plotly_figure(fig, "decoder_representation_heatmap.html")
fig.show()

embedding_weights = model.encoder.embedding.get_weights()[0]
non_special_tokens = [(token, token_id) for token, token_id in source_tokenizer.token_to_id.items()
                      if token not in SPECIAL_TOKENS and token_id < len(embedding_weights)]
selected_embedding_tokens = sorted(non_special_tokens, key=lambda item: source_tokenizer.frequencies.get(item[0], 0), reverse=True)[:80]
if len(selected_embedding_tokens) >= 3:
    selected_vectors = np.array([embedding_weights[token_id] for _, token_id in selected_embedding_tokens])
    coordinates = PCA(n_components=2, random_state=SEED).fit_transform(selected_vectors)
    pca_frame = pd.DataFrame({"token": [token for token, _ in selected_embedding_tokens], "PC1": coordinates[:, 0], "PC2": coordinates[:, 1],
                              "frequency": [source_tokenizer.frequencies.get(token, 0) for token, _ in selected_embedding_tokens]})
    fig = px.scatter(pca_frame, x="PC1", y="PC2", text="token", size="frequency", hover_name="token",
                     title="PCA projection of learned source-token embeddings", color="frequency", color_continuous_scale="Viridis")
    fig.update_traces(textposition="top center")
    fig.update_layout(template="plotly_white", height=650)
    save_plotly_figure(fig, "embedding_pca.html")
    fig.show()

### Parameter counts by module

The chart below groups trainable parameters by where they live in the custom implementation. Positional encodings are deterministic constants and contribute no trainable parameters. Layer normalization, embeddings, attention projections, feed-forward dense layers, and the vocabulary projection are trainable.

In [28]:
def count_trainable_parameters(layer):
    return int(sum(np.prod(variable.shape) for variable in layer.trainable_variables))

parameter_groups = {
    "Source embedding": count_trainable_parameters(model.encoder.embedding),
    "Encoder attention": sum(count_trainable_parameters(block.self_attention) for block in model.encoder.blocks),
    "Encoder FFN": sum(count_trainable_parameters(block.feed_forward) for block in model.encoder.blocks),
    "Encoder norms": sum(count_trainable_parameters(block.norm_attention) + count_trainable_parameters(block.norm_ffn)
                          for block in model.encoder.blocks),
    "Target embedding": count_trainable_parameters(model.decoder.embedding),
    "Decoder self-attention": sum(count_trainable_parameters(block.self_attention) for block in model.decoder.blocks),
    "Decoder cross-attention": sum(count_trainable_parameters(block.cross_attention) for block in model.decoder.blocks),
    "Decoder FFN": sum(count_trainable_parameters(block.feed_forward) for block in model.decoder.blocks),
    "Decoder norms": sum(count_trainable_parameters(block.norm_self) + count_trainable_parameters(block.norm_cross) + count_trainable_parameters(block.norm_ffn)
                          for block in model.decoder.blocks),
    "Vocabulary projection": count_trainable_parameters(model.output_projection),
}
parameter_frame = pd.DataFrame({"module": list(parameter_groups.keys()), "parameters": list(parameter_groups.values())})
parameter_frame.to_csv(OUTPUT_DIRS["metrics"] / "parameter_counts.csv", index=False)
print("Total trainable parameters:", model.count_params())
print("Grouped parameter sum:", int(parameter_frame.parameters.sum()))
assert int(parameter_frame.parameters.sum()) == model.count_params()

display(parameter_frame.sort_values("parameters", ascending=False))
fig = px.bar(parameter_frame.sort_values("parameters"), x="parameters", y="module", orientation="h",
             color="parameters", color_continuous_scale="Blues", title="Trainable parameters by module")
fig.update_layout(template="plotly_white", height=600)
save_plotly_figure(fig, "parameter_distribution.html")
fig.show()

Total trainable parameters: 123530
Grouped parameter sum: 123530


,module,parameters
1,Encoder attention,16640
5,Decoder self-attention,16640
6,Decoder cross-attention,16640
7,Decoder FFN,16576
2,Encoder FFN,16576
0,Source embedding,13760
9,Vocabulary projection,13130
4,Target embedding,12928
8,Decoder norms,384
3,Encoder norms,256


## Optional ablation experiments

Ablations ask whether a design choice matters under a controlled training budget. The runnable function below compares a small baseline, one attention head, an additional encoder/decoder layer, and positional encoding disabled. The positional ablation changes the task because order-sensitive instructions lose an explicit position signal.

To keep the default end-to-end notebook CPU-friendly, the optional ablation run is **disabled by default**. Turn on `RUN_ABLATIONS` to train each variant on the same small subset for the same number of epochs and score each on the same held-out subset. Do not treat these shortened experiments as a definitive architecture ranking; a fairer study would repeat seeds and equalize compute budgets more carefully.

In [29]:
def run_ablation_experiments(train_limit=2048, validation_limit=256, test_limit=128, epochs=2):
    variants = [
        {"name": "baseline_4_heads", "num_heads": 4, "num_encoder_layers": 1, "num_decoder_layers": 1, "d_ff": 128, "use_positional_encoding": True},
        {"name": "single_head", "num_heads": 1, "num_encoder_layers": 1, "num_decoder_layers": 1, "d_ff": 128, "use_positional_encoding": True},
        {"name": "two_layers", "num_heads": 4, "num_encoder_layers": 2, "num_decoder_layers": 2, "d_ff": 128, "use_positional_encoding": True},
        {"name": "no_positional_encoding", "num_heads": 4, "num_encoder_layers": 1, "num_decoder_layers": 1, "d_ff": 128, "use_positional_encoding": False},
    ]
    train_ds = make_tf_dataset(train_src[:train_limit], train_tgt_in[:train_limit], train_labels[:train_limit], 64, shuffle=True)
    val_ds = make_tf_dataset(val_src[:validation_limit], val_tgt_in[:validation_limit], val_labels[:validation_limit], 64, shuffle=False)
    rows = []

    for variant in variants:
        variant_model = Transformer(
            len(source_tokenizer.token_to_id), len(target_tokenizer.token_to_id),
            d_model=64, num_heads=variant["num_heads"],
            num_encoder_layers=variant["num_encoder_layers"], num_decoder_layers=variant["num_decoder_layers"],
            d_ff=variant["d_ff"], dropout_rate=CONFIG["dropout"],
            source_max_length=SRC_MAX_LEN, target_max_length=TGT_MAX_LEN,
            use_positional_encoding=variant["use_positional_encoding"], name=variant["name"]
        )
        _ = variant_model((tf.constant(train_src[:1]), tf.constant(train_tgt_in[:1])), training=False)
        variant_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001, clipnorm=1.0)

        @tf.function(reduce_retracing=True)
        def variant_train_step(source_batch, target_input_batch, target_label_batch):
            with tf.GradientTape() as tape:
                logits = variant_model((source_batch, target_input_batch), training=True)
                loss_value = masked_sparse_cross_entropy(target_label_batch, logits)
            grads = tape.gradient(loss_value, variant_model.trainable_variables)
            pairs = [(g, v) for g, v in zip(grads, variant_model.trainable_variables) if g is not None]
            variant_optimizer.apply_gradients(pairs)
            return loss_value

        start_time = time.perf_counter()
        for _ in range(epochs):
            for source_batch, target_input_batch, target_label_batch in train_ds:
                variant_train_step(source_batch, target_input_batch, target_label_batch)
        training_seconds = time.perf_counter() - start_time
        val_losses = []
        for source_batch, target_input_batch, target_label_batch in val_ds:
            logits = variant_model((source_batch, target_input_batch), training=False)
            val_losses.append(float(masked_sparse_cross_entropy(target_label_batch, logits).numpy()))
        mini_eval, mini_metrics = evaluate_autoregressively(variant_model, test_frame.iloc[:test_limit].reset_index(drop=True),
                                                            test_src[:test_limit], test_tgt_full[:test_limit],
                                                            batch_size=32, limit=test_limit)
        rows.append({"variant": variant["name"], "parameters": int(variant_model.count_params()),
                     "validation_loss": float(np.mean(val_losses)), "exact_match_accuracy": mini_metrics["exact_match_accuracy"],
                     "training_seconds": training_seconds, "epochs": epochs, "train_examples": min(train_limit, len(train_src)),
                     "test_examples": mini_metrics["evaluated_examples"]})
        del variant_model, variant_optimizer

    results = pd.DataFrame(rows)
    results.to_csv(OUTPUT_DIRS["metrics"] / "ablation_results.csv", index=False)
    return results


RUN_ABLATIONS = False
if RUN_ABLATIONS:
    ablation_results = run_ablation_experiments()
    display(ablation_results)
    fig = px.bar(ablation_results, x="variant", y="exact_match_accuracy", color="variant",
                 title="Ablation results: exact match on a fixed held-out subset")
    fig.update_layout(template="plotly_white", showlegend=False, xaxis_tickangle=-20, yaxis_range=[0, 1])
    save_plotly_figure(fig, "ablation_exact_match.html")
    fig.show()
else:
    print("Ablations are implemented but were not run. Set RUN_ABLATIONS = True to execute and create measured results.")

Ablations are implemented but were not run. Set RUN_ABLATIONS = True to execute and create measured results.


## Tests and revision sheet

The assertions here are intended to catch structural errors early. They test positional-encoding shapes, attention dimensions and normalization, causal masking, padding masking, encoder/decoder output shapes, full logits, padding-aware loss, gradient finiteness, checkpoint restoration, and autoregressive termination. Passing these checks does not prove the model has learned every task; it verifies important implementation contracts.

In [30]:
# Positional encoding dimensions and a known value at position zero.
pe_test = sinusoidal_position_encoding(12, 8).numpy()
assert pe_test.shape == (1, 12, 8)
assert np.allclose(pe_test[0, 0, 0::2], 0.0, atol=1e-7)
assert np.allclose(pe_test[0, 0, 1::2], 1.0, atol=1e-7)

# Attention output dimensions and probability normalization.
tiny_q = tf.random.normal([2, 3, 4], seed=SEED)
tiny_k = tf.random.normal([2, 5, 4], seed=SEED + 1)
tiny_v = tf.random.normal([2, 5, 6], seed=SEED + 2)
tiny_out, tiny_weights, _, _ = scaled_dot_product_attention(tiny_q, tiny_k, tiny_v)
assert tiny_out.shape == (2, 3, 6)
assert np.allclose(tiny_weights.numpy().sum(axis=-1), 1.0, atol=1e-6)

# Causal mask: all entries strictly above the diagonal must be masked.
causal_test = create_look_ahead_mask(6).numpy()[0, 0]
assert np.all(causal_test[np.triu_indices(6, k=1)])
assert not np.any(np.tril(causal_test))

# Padding positions are marked; valid positions remain open.
padding_test = create_padding_mask(tf.constant([[4, 5, PAD_ID, PAD_ID]])).numpy()[0, 0, 0]
assert padding_test.tolist() == [False, False, True, True]

# Multi-head dimensions and attention distribution.
shape_layer = MultiHeadAttention(16, 4)
shape_tensor = tf.random.normal([2, 7, 16], seed=SEED)
shape_output, shape_attention = shape_layer(shape_tensor, shape_tensor, shape_tensor)
assert shape_output.shape == (2, 7, 16)
assert shape_attention.shape == (2, 4, 7, 7)
assert np.allclose(shape_attention.numpy().sum(axis=-1), 1.0, atol=1e-5)

# Full model, loss, and gradient checks.
small_src = tf.constant(train_src[:3])
small_tgt_in = tf.constant(train_tgt_in[:3])
small_labels = tf.constant(train_labels[:3])
with tf.GradientTape() as tape:
    small_logits = model((small_src, small_tgt_in), training=True)
    small_loss = masked_sparse_cross_entropy(small_labels, small_logits)
small_grads = tape.gradient(small_loss, model.trainable_variables)
assert small_logits.shape == (3, train_tgt_in.shape[1], V_TARGET)
assert np.isfinite(small_loss.numpy())
assert any(gradient is not None for gradient in small_grads)

# Check finiteness while handling both standard Tensors and IndexedSlices
for gradient in small_grads:
    if gradient is not None:
        grad_val = gradient.values if isinstance(gradient, tf.IndexedSlices) else gradient
        assert np.isfinite(grad_val.numpy()).all()

# Padding positions should not affect the masked average of loss.
labels_base = tf.constant([[4, 5, PAD_ID]], dtype=tf.int32)
logits_base = tf.zeros([1, 3, V_TARGET], dtype=tf.float32)
labels_with_extra_pad = tf.constant([[4, 5, PAD_ID, PAD_ID]], dtype=tf.int32)
logits_with_extra_pad = tf.zeros([1, 4, V_TARGET], dtype=tf.float32)
assert np.allclose(masked_sparse_cross_entropy(labels_base, logits_base).numpy(),
                   masked_sparse_cross_entropy(labels_with_extra_pad, logits_with_extra_pad).numpy(), atol=1e-6)

# Checkpoint exists and can be restored into the current built model.
assert checkpoint_path.exists(), "Best-validation checkpoint was not created during training."
model.load_weights(checkpoint_path)

# Autoregressive generation starts with BOS and never exceeds the configured maximum.
smoke_generated = greedy_decode_batch(model, tf.constant(test_src[:2]), max_new_tokens=min(5, TGT_MAX_LEN - 1))
assert smoke_generated.shape[0] == 2
assert smoke_generated.shape[1] <= min(5, TGT_MAX_LEN - 1) + 1
assert np.all(smoke_generated.numpy()[:, 0] == BOS_ID)

print("All structural tests passed.")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'masked_self_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning:

Layer 'cross_attention' (of type MultiHeadAttention) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.



All structural tests passed.


### Mini-batch overfit sanity check

A model should be able to lower training loss on a very small fixed batch if the forward pass, labels, masking, and gradients are wired correctly. This is a diagnostic—not a test score and not proof of generalization. The function is provided for a controlled experiment because running it mutates model weights; it is disabled by default so it does not silently change the main trained checkpoint.

In [31]:
def run_small_batch_overfit_check(source_batch, target_input_batch, label_batch, steps=20, learning_rate=1e-3):
    sanity_model = Transformer(
        len(source_tokenizer.token_to_id), len(target_tokenizer.token_to_id),
        d_model=32, num_heads=4, num_encoder_layers=1, num_decoder_layers=1, d_ff=64,
        dropout_rate=0.0, source_max_length=SRC_MAX_LEN, target_max_length=TGT_MAX_LEN,
    )
    _ = sanity_model((source_batch, target_input_batch), training=False)
    sanity_optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    values = []
    for _ in range(steps):
        with tf.GradientTape() as tape:
            logits = sanity_model((source_batch, target_input_batch), training=True)
            loss_value = masked_sparse_cross_entropy(label_batch, logits)
        gradients = tape.gradient(loss_value, sanity_model.trainable_variables)
        sanity_optimizer.apply_gradients(zip(gradients, sanity_model.trainable_variables))
        values.append(float(loss_value.numpy()))
    return sanity_model, pd.DataFrame({"step": np.arange(1, steps + 1), "loss": values})


RUN_OVERFIT_SANITY_CHECK = False
if RUN_OVERFIT_SANITY_CHECK:
    sanity_model, sanity_history = run_small_batch_overfit_check(
        tf.constant(train_src[:8]), tf.constant(train_tgt_in[:8]), tf.constant(train_labels[:8])
    )
    display(sanity_history.head())
    fig = px.line(sanity_history, x="step", y="loss", markers=True, title="Small-batch overfit sanity check")
    fig.update_layout(template="plotly_white")
    save_plotly_figure(fig, "small_batch_overfit_check.html")
    fig.show()
else:
    print("Small-batch overfit check is implemented but disabled to avoid an extra training run by default.")

Small-batch overfit check is implemented but disabled to avoid an extra training run by default.


## Save artifacts and inspect outputs

The notebook writes all output files relative to `transformer_from_scratch_outputs/`. Interactive Plotly figures are written as HTML, metrics and prediction tables as CSV/JSON, and the best validation weights as a checkpoint. Optional static Plotly PNG export is attempted only when the rendering dependency is installed; HTML remains the primary interactive artifact.

In [32]:
with open(OUTPUT_DIRS["model"] / "source_vocabulary.json", "w", encoding="utf-8") as file:
    json.dump(source_tokenizer.to_dict(), file, ensure_ascii=False, indent=2)
with open(OUTPUT_DIRS["model"] / "target_vocabulary.json", "w", encoding="utf-8") as file:
    json.dump(target_tokenizer.to_dict(), file, ensure_ascii=False, indent=2)
with open(OUTPUT_DIRS["model"] / "model_config.json", "w", encoding="utf-8") as file:
    json.dump({
        "source_vocab_size": len(source_tokenizer.token_to_id),
        "target_vocab_size": len(target_tokenizer.token_to_id),
        "d_model": CONFIG["d_model"], "num_heads": CONFIG["num_heads"],
        "num_encoder_layers": CONFIG["num_encoder_layers"], "num_decoder_layers": CONFIG["num_decoder_layers"],
        "d_ff": CONFIG["d_ff"], "dropout_rate": CONFIG["dropout"],
        "source_max_length": SRC_MAX_LEN, "target_max_length": TGT_MAX_LEN,
    }, file, indent=2)

dataset.to_csv(OUTPUT_DIRS["data"] / "synthetic_dataset.csv", index=False)
split_summary.to_csv(OUTPUT_DIRS["data"] / "split_summary.csv", index=False)

plotly_files = sorted(OUTPUT_DIRS["plots_html"].glob("*.html"))
image_files = sorted(OUTPUT_DIRS["images"].glob("*"))
summary = {
    "dataset_examples": len(dataset),
    "train_examples": len(train_frame),
    "validation_examples": len(validation_frame),
    "test_examples": len(test_frame),
    "trainable_parameters": int(model.count_params()),
    "plots_html_count": len(plotly_files),
    "static_image_count": len(image_files),
    "checkpoint": checkpoint_path.as_posix(),
    "outputs_root": PROJECT_DIR.as_posix(),
}
with open(OUTPUT_DIRS["metrics"] / "artifact_summary.json", "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)
print(json.dumps(summary, indent=2))
print("HTML plots:")
for path in plotly_files:
    print(" -", path.relative_to(PROJECT_DIR).as_posix())

{
  "dataset_examples": 10400,
  "train_examples": 8320,
  "validation_examples": 1040,
  "test_examples": 1040,
  "trainable_parameters": 123530,
  "plots_html_count": 37,
  "static_image_count": 0,
  "checkpoint": "transformer_from_scratch_outputs/checkpoints/best_transformer.weights.h5",
  "outputs_root": "transformer_from_scratch_outputs"
}
HTML plots:
 - plots_html/architecture_diagram.html
 - plots_html/attention_probabilities.html
 - plots_html/attention_raw_scores.html
 - plots_html/attention_scaled_scores.html
 - plots_html/attention_scaling_comparison.html
 - plots_html/attention_weighted_values.html
 - plots_html/cross_attention_head_1.html
 - plots_html/cross_attention_head_2.html
 - plots_html/cross_attention_head_3.html
 - plots_html/cross_attention_head_4.html
 - plots_html/dataset_task_distribution.html
 - plots_html/decoder_representation_heatmap.html
 - plots_html/decoder_self_attention_head_1.html
 - plots_html/decoder_self_attention_head_2.html
 - plots_html/decoder

## Final revision sheet: equations and data flow

### Core equations

**Scaled dot-product attention**

\[
\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}+M\right)V
\]

**Multi-head attention**

\[
\operatorname{MHA}(Q,K,V)=\operatorname{Concat}(head_1,\ldots,head_H)W^O
\]

where each head uses learned projections: `head_i = Attention(QW_i^Q, KW_i^K, VW_i^V)`.

**Sinusoidal position encoding**

\[
PE(pos,2i)=\sin(pos/10000^{2i/D}),\quad PE(pos,2i+1)=\cos(pos/10000^{2i/D})
\]

**Position-wise feed-forward network**

\[
FFN(x)=W_2\operatorname{ReLU}(W_1x+b_1)+b_2
\]

**Post-normalization residual block**

\[
y=\operatorname{LayerNorm}(x+\operatorname{Sublayer}(x))
\]

**Masked token loss**

\[
L=\frac{\sum_{b,t}\mathbf{1}[y_{bt}\ne PAD]\,\operatorname{CE}(y_{bt},z_{bt})}
{\sum_{b,t}\mathbf{1}[y_{bt}\ne PAD]}
\]

### End-to-end flow

1. Tokenize source/target text and map tokens to integer IDs.
2. Add `[EOS]` to source; add `[BOS]` and `[EOS]` to target.
3. Shift target right to produce decoder inputs and next-token labels.
4. Embed tokens and add positional encodings.
5. Encoder self-attention contextualizes source tokens while masking source padding.
6. Decoder masked self-attention reads only earlier/current target positions.
7. Decoder cross-attention queries encoder memory while respecting source padding.
8. Feed-forward layers transform each position independently, with residual connections and layer normalization.
9. A linear projection produces vocabulary logits; padded labels are ignored by the loss.
10. At inference, greedily append tokens until `[EOS]` or the maximum length.

### Glossary

- **Query (Q):** projected representation used to ask which keys are relevant.
- **Key (K):** projected representation compared with a query.
- **Value (V):** projected content mixed according to attention weights.
- **Attention score:** query–key compatibility before softmax.
- **Attention probability:** normalized score assigned to a key position.
- **Head:** one learned attention projection pathway.
- **Positional encoding:** deterministic vector added to represent token location.
- **Encoder memory:** contextualized output of the encoder stack.
- **Cross-attention:** decoder queries attending over encoder memory.
- **Causal mask:** prevents attention to future target positions.
- **Teacher forcing:** provide the ground-truth previous token during training.
- **Autoregressive decoding:** feed generated tokens back in to generate the next token.
- **Logits:** unnormalized vocabulary scores.
- **Padding-aware loss:** loss that ignores labels used only to pad sequences.
- **Gradient clipping:** limit gradient norm to reduce damaging parameter updates.
- **Overfitting:** good training performance paired with poorer validation/test behavior.

## Conceptual questions with answers

**Why is attention scaled by the square root of the key dimension?** Dot products tend to grow in magnitude as feature width grows. Scaling helps keep softmax from becoming unnecessarily saturated.

**Why use multiple heads?** Separate projections can capture different relationships in parallel. Meaningful, distinct roles are possible but not guaranteed.

**Why does the encoder need positional information?** Attention alone does not inherently encode the absolute order of tokens, while many tasks depend on order.

**What does the source padding mask do?** It prevents padded source positions from receiving attention as keys/values.

**What does a causal mask do?** It prevents decoder position `t` from reading target positions later than `t`.

**How are self-attention and cross-attention different?** Self-attention uses one sequence for queries, keys, and values; cross-attention uses decoder states as queries and encoder memory as keys and values.

**Why are residual connections used?** They give information and gradients a direct path around a sublayer.

**What does layer normalization normalize?** It normalizes features within each individual token representation, followed by learned scale and offset.

**Why shift the target sequence?** The decoder input contains a prefix and the label is the next token at each position.

**Why is softmax not applied before the cross-entropy operation here?** TensorFlow's sparse softmax-cross-entropy expects logits and performs the stable log-softmax calculation internally.

**Why ignore `[PAD]` in loss?** Padding is a batching convenience, not a real prediction target.

**Why can training parallelize target positions while inference cannot?** Causal masking makes all shifted training positions valid simultaneously; inference does not know future output tokens.

**What does exact-match accuracy measure?** It counts an example correct only if the whole generated response matches the reference, making it stricter than token accuracy.

**Why is greedy decoding imperfect?** The locally highest-probability token does not necessarily lead to the highest-probability complete sequence.

**What are the limits of this benchmark?** It uses controlled templates, a small vocabulary, and short responses; results do not establish broad language ability or real-world translation quality.

## Coding exercises

1. **Head inspection:** choose a different test example and compare encoder self-attention heads. Do not assign meanings to heads until you have inspected multiple examples.
2. **Mask experiment:** deliberately remove the causal mask in a separate experimental copy. Explain why training behavior becomes invalid for autoregressive next-token prediction.
3. **Capacity sweep:** train `d_model=32`, `64`, and `128` with an otherwise fixed setup. Record parameters, training time, and held-out per-task exact match.
4. **Tokenizer change:** replace whitespace/regex tokenization with a subword tokenizer you implement or study. Compare vocabulary size, unknown rate, sequence length, and memory cost.
5. **Decoding change:** implement beam search and compare outputs and latency against greedy decoding. Keep evaluation examples identical.

## Practical next steps

Adapt the pipeline to a legitimate parallel translation dataset, maintain train/validation/test separation, and avoid building the vocabulary on evaluation text. Add subword tokenization to handle open vocabulary, experiment with beam search, and scale model width/depth only after the shape and masking tests pass. For larger datasets, consider optimized attention kernels and training infrastructure—but keep a small reference implementation for debugging.

## Final caveats

This notebook is designed to execute from top to bottom in a compatible TensorFlow environment. Training and all model-derived figures are generated when the notebook is run; they are not pre-filled with fabricated metrics. Optional ablation and tiny-batch overfit experiments are disabled by default to avoid adding extra training time. For a full-scale ablation study, run the variants with repeated seeds and matched compute budgets.